# Predicción del caudal medio diario en cuencas de Colombia con una LSTM regional

**Proyecto final — Análisis con Deep Learning, Universidad de los Andes**
Santiago Palacios Loaiza · Diego Leonardo Barón Espitia · Andrés Felipe Mosquera Hernández · Juan Sebastián Rodríguez Avella

**Parte 2: modelado.** Este notebook parte de la base que genera `proyecto-final-preprocesamiento.ipynb` en `data/processed/`:

1. **Base para el modelo:** lee la base, verifica que esté completa y crea las listas de cuencas.
2. **Configuración del modelo:** define y explica cada hiperparámetro.
3. **Prueba de humo:** entrenamiento corto (20 cuencas, 2 épocas) para verificar que todo funciona, y prueba del tamaño de lote.
4. **Estrategia:** los dos flujos de evaluación (cuenca vista y cuenca no vista), las etapas y el tiempo de cómputo.
5. **Anidamiento y particiones espaciales:** sistemas de ríos y las 5 particiones de la validación cruzada.
6. **Funciones de entrenamiento y evaluación:** entrenar, retomar, elegir la mejor época y calcular métricas.
7. **Etapa 1 (opcional):** ajuste de `hidden_size`; el equipo lo fijó en 256.
8. **Etapa 2, flujo temporal:** el modelo principal, evaluado en 2016–2022. En 8.4 a 8.7: selección de variables con ERA5-Land, confirmación con el LSTM y evaluación del modelo final.
9. **Etapa 3, flujo espacial:** validación cruzada en 5 particiones.
10. **Baselines** y 11. **Resultados** frente a las metas.
12. **Fase auxiliar:** transfer learning con embeddings preentrenados, si alcanza el tiempo.

> **Regla del proyecto:** el modelo **no** recibe caudal observado como entrada (ni `Q(t-1)` ni nada derivado del caudal).
> Es simulación a partir del clima, y eso es lo que permite usarlo en cuencas sin estación de aforo.

## 0. Entorno

In [1]:
import hashlib
import logging
import pickle
import re
import shutil
import struct
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xarray as xr
import yaml
from IPython.display import display
from matplotlib.path import Path as Trazo
from scipy.sparse.csgraph import connected_components

ROOT = Path.cwd()
DIR_BASE = ROOT / "data" / "processed"   # base generada por proyecto-final-preprocesamiento.ipynb
assert (DIR_BASE / "time_series").exists(), "Ejecuta primero proyecto-final-preprocesamiento.ipynb"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

**Instalación (una sola vez).** PyTorch debe tener soporte CUDA para entrenar en la GPU. Si ya hay un PyTorch solo-CPU instalado, pip no lo reemplaza: hay que forzar la reinstalación (`--force-reinstall`) con el kernel recién reiniciado. La celda siguiente debe mostrar `True` y el nombre de la GPU.

In [2]:
#import sys
#!{sys.executable} -m pip install neuralhydrology
#!{sys.executable} -m pip install lightgbm   # sección 8.3
#!{sys.executable} -m pip install torch==2.13.0 torchvision==0.28.0 --index-url https://download.pytorch.org/whl/cu126 --force-reinstall --no-deps


In [3]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.13.0+cu126 True NVIDIA GeForce RTX 2060 with Max-Q Design


## 1. Base para el modelo

El notebook de preprocesamiento deja en `data/processed/`:

| Archivo | Contenido | Uso |
|---|---|---|
| `time_series/<cuenca>.nc` | Forzantes y caudal en mm/día, 1981–2022, una serie por cuenca | NeuralHydrology, al entrenar y evaluar |
| `attributes/atributos_modelo.csv` | Los 48 atributos estáticos | NeuralHydrology |
| `camels_col_diario.parquet` | Tabla única: lo anterior más el período, el estado del caudal y la marca de interpolación | Períodos y cobertura (1.1); análisis de resultados |
| `resumen_cuencas.csv` | Región y días con caudal por período de cada cuenca | Listas de cuencas (1.2) |

Las variables y las fechas de la configuración (sección 2) se toman de estos archivos, no se escriben a mano: el modelo usa exactamente lo que dejó el preprocesamiento. Si este cambia, basta con volver a ejecutar este notebook.

### 1.1 Lectura de la base

Las variables salen de los archivos que lee NeuralHydrology y los períodos, de la tabla. Se verifica que todos los archivos describan las mismas cuencas y variables.

In [4]:
OBJETIVO = "streamflow"   # caudal en mm/día

# Variables del modelo: las que NeuralHydrology encontrará en los archivos
archivos_nc = sorted((DIR_BASE / "time_series").glob("*.nc"))
with xr.open_dataset(archivos_nc[0]) as ds:
    FORZANTES = [v for v in ds.data_vars if v != OBJETIVO]
atributos_modelo = pd.read_csv(DIR_BASE / "attributes" / "atributos_modelo.csv",
                               dtype={"gauge_id": str}, index_col="gauge_id")
ATRIBUTOS = atributos_modelo.columns.tolist()

# Tabla diaria (solo las columnas que se usan aquí) y resumen por cuenca
RUTA_TABLA = DIR_BASE / "camels_col_diario.parquet"
diario = pd.read_parquet(RUTA_TABLA, columns=["gauge_id", "date", "periodo", OBJETIVO])
resumen = pd.read_csv(DIR_BASE / "resumen_cuencas.csv", dtype={"gauge_id": str})
cuencas = resumen["gauge_id"].tolist()

# Períodos: primer y último día de cada etiqueta de la tabla (partición temporal del preprocesamiento, 1.9)
PERIODOS = {p: (diario.loc[diario["periodo"] == p, "date"].min(), diario.loc[diario["periodo"] == p, "date"].max())
            for p in ["train", "validation", "test"]}

# Todos los archivos deben describir las mismas cuencas y variables
assert [a.stem for a in archivos_nc] == sorted(cuencas), "Las series .nc no coinciden con resumen_cuencas.csv"
assert sorted(atributos_modelo.index) == sorted(cuencas), "atributos_modelo.csv no coincide con resumen_cuencas.csv"
assert set(diario["gauge_id"].astype(str)) == set(cuencas), "La tabla diaria no coincide con resumen_cuencas.csv"
assert set(FORZANTES + ATRIBUTOS) <= set(pq.read_schema(RUTA_TABLA).names), "Faltan variables en la tabla diaria"
assert not atributos_modelo.isna().any().any(), "Hay atributos vacíos"

print(f"{len(cuencas)} cuencas | {len(FORZANTES)} forzantes + {len(ATRIBUTOS)} atributos = "
      f"{len(FORZANTES) + len(ATRIBUTOS)} entradas por día")
print("Forzantes:", FORZANTES)

248 cuencas | 4 forzantes + 48 atributos = 52 entradas por día
Forzantes: ['pr', 'poten_evapo', 't_min', 't_max']


In [5]:
# Cobertura de cada período
con_caudal = diario[diario[OBJETIVO].notna()]
pd.DataFrame({
    "inicio": [PERIODOS[p][0].date() for p in PERIODOS],
    "fin": [PERIODOS[p][1].date() for p in PERIODOS],
    "cuencas con caudal": [con_caudal.loc[con_caudal["periodo"] == p, "gauge_id"].nunique() for p in PERIODOS],
    "días con caudal": [int((con_caudal["periodo"] == p).sum()) for p in PERIODOS],
}, index=list(PERIODOS))

,inicio,fin,cuencas con caudal,días con caudal
train,1981-01-01,2010-12-31,248,2267252
validation,2011-01-01,2015-12-31,213,320301
test,2016-01-01,2022-12-31,189,388971


### 1.2 Listas de cuencas

NeuralHydrology recibe las cuencas de cada fase en un archivo de texto, un id por línea:
- `configs/basins/todas.txt`: las 248 cuencas, para entrenar el modelo principal.
- `configs/basins/humo_20.txt`: 20 cuencas para la prueba de humo, 4 por región elegidas al azar entre las que tienen caudal en los tres períodos.
- `configs/basins/validacion.txt` y `configs/basins/prueba.txt`: las cuencas con al menos un año de días evaluables en 2011–2015 y en 2016–2022.

Un día es **evaluable** si tiene caudal observado y los 365 días anteriores tienen las 4 forzantes completas: solo en esos días el LSTM produce una simulación que se puede comparar con el caudal. Para validar o evaluar una cuenca se exige al menos un año de días evaluables en el período (`MIN_DIAS_EVALUACION`), por dos razones:
- Con pocos días, el NSE de una cuenca no es confiable.
- NeuralHydrology falla al calcular FLV y FMS en una cuenca con un solo día, y hay una así en validación. Sin este filtro, la primera validación del modelo principal se detendría con un error.

Las cuencas con menos datos igual entrenan.

In [6]:
DIR_CUENCAS = ROOT / "configs" / "basins"
DIR_CUENCAS.mkdir(parents=True, exist_ok=True)
(DIR_CUENCAS / "todas.txt").write_text("\n".join(resumen["gauge_id"]) + "\n")

con_tres_periodos = resumen[(resumen["dias_q_train"] > 0) & (resumen["dias_q_validation"] > 0) & (resumen["dias_q_test"] > 0)]
humo = con_tres_periodos.groupby("region").sample(n=4, random_state=42).sort_values(["region", "gauge_id"])
(DIR_CUENCAS / "humo_20.txt").write_text("\n".join(humo["gauge_id"]) + "\n")
humo[["gauge_id", "region", "dias_q_train", "dias_q_validation", "dias_q_test"]]

,gauge_id,region,dias_q_train,dias_q_validation,dias_q_test
217,42067010,Amazon,6927,1643,1825
218,42077010,Amazon,7279,1775,1786
219,42077020,Amazon,7539,1816,1818
223,47017100,Amazon,10341,1818,1461
26,15067150,Caribbean,681,1627,2080
27,16017020,Caribbean,10770,1766,2552
30,16027280,Caribbean,5897,1656,1829
40,16057040,Caribbean,10508,1774,731
65,21197010,Magdalena-Cauca,10709,1818,2471
111,23147020,Magdalena-Cauca,10668,1795,2470


In [7]:
# Días evaluables por cuenca y período: con caudal y con la ventana de 365 días de forzantes completa
LARGO_VENTANA = 365          # seq_length (2.3)
MIN_DIAS_EVALUACION = 365    # mínimo de días evaluables para validar o evaluar una cuenca

tabla_dias = pd.read_parquet(RUTA_TABLA, columns=["gauge_id", "date", "periodo", OBJETIVO] + FORZANTES)
tabla_dias["gauge_id"] = tabla_dias["gauge_id"].astype(str)
tabla_dias = tabla_dias.sort_values(["gauge_id", "date"])
forzantes_completas = tabla_dias[FORZANTES].notna().all(axis=1)
ventana_completa = (forzantes_completas.groupby(tabla_dias["gauge_id"])
                    .transform(lambda s: s.rolling(LARGO_VENTANA).sum() == LARGO_VENTANA))
dias_evaluables = ((ventana_completa & tabla_dias[OBJETIVO].notna())
                   .groupby([tabla_dias["gauge_id"], tabla_dias["periodo"]], observed=True).sum().unstack()[list(PERIODOS)])
del tabla_dias

validan = dias_evaluables.index[dias_evaluables["validation"] >= MIN_DIAS_EVALUACION].tolist()
se_evaluan = dias_evaluables.index[dias_evaluables["test"] >= MIN_DIAS_EVALUACION].tolist()
(DIR_CUENCAS / "validacion.txt").write_text("\n".join(validan) + "\n")
(DIR_CUENCAS / "prueba.txt").write_text("\n".join(se_evaluan) + "\n")

print(f"validacion.txt: {len(validan)} cuencas | prueba.txt: {len(se_evaluan)} cuencas")
pd.DataFrame({"con algún día evaluable": (dias_evaluables > 0).sum(),
              "con menos de un año": ((dias_evaluables > 0) & (dias_evaluables < MIN_DIAS_EVALUACION)).sum(),
              "con al menos un año": (dias_evaluables >= MIN_DIAS_EVALUACION).sum()}).T

validacion.txt: 195 cuencas | prueba.txt: 183 cuencas


periodo,train,validation,test
con algún día evaluable,248,210,188
con menos de un año,1,15,5
con al menos un año,247,195,183


## 2. Configuración del modelo

La configuración se define en Python (2.5) y se escribe en `configs/base.yml`, que es lo que lee NeuralHydrology.

### 2.1 Datos y períodos

| Parámetro | Valor | Qué hace |
|---|---|---|
| `dataset`, `data_dir` | `generic`, `data/processed` | Lector genérico de NeuralHydrology, con los archivos de la sección 1 |
| `train_basin_file` | `configs/basins/todas.txt` | Las 248 cuencas (flujo temporal). Las etapas 1 y 3 cambian las listas (sección 4) |
| `validation_basin_file`, `test_basin_file` | `validacion.txt` (195), `prueba.txt` (183) | Las cuencas con al menos un año de días evaluables en cada período (1.2) |
| Fechas de inicio y fin | 1981–2010, 2011–2015, 2016–2022 | Partición temporal del preprocesamiento, leída de la tabla (1.1), en formato `DD/MM/YYYY` |
| `dynamic_inputs` | `pr`, `poten_evapo`, `t_min`, `t_max` | Forzantes que el modelo lee cada día de la ventana |
| `static_attributes` | 48 atributos | Se repiten cada día junto a las forzantes: 52 entradas por día |
| `target_variables` | `streamflow` | Caudal en mm/día |
| `clip_targets_to_zero` | `streamflow` | Las predicciones negativas se llevan a 0 al calcular las métricas |

### 2.2 Arquitectura

| Parámetro | Valor | Qué hace | Origen |
|---|---|---|---|
| `model` | `cudalstm` | LSTM de una capa, optimizado para GPU | Propuesta |
| `hidden_size` | 256 | Número de unidades (neuronas) de la capa oculta del LSTM. Fijado en 256 por el equipo (11/11/2026); el ajuste de la sección 7 queda opcional | (2025) |
| `initial_forget_bias` | 3 | La compuerta de olvido arranca casi abierta (σ(3) ≈ 0,95), lo que ayuda a aprender dependencias largas como el flujo base | Ejemplos de NeuralHydrology |
| `output_dropout` | 0,4 | Apaga al azar el 40 % de las salidas del LSTM durante el entrenamiento, para reducir el sobreajuste | |
| `head`, `output_activation` | `regression`, `linear` | Capa final lineal: convierte los 256 estados en un valor de caudal | |

**Qué es `hidden_size`.** Es el número de unidades de la capa oculta del LSTM, que tiene una sola capa. Cada unidad es una neurona con memoria: cada día, el LSTM actualiza dos vectores de 256 valores, el estado oculto *h*, que pasa a la capa final, y la memoria *c*, que puede conservar por meses información como cuánta agua almacena la cuenca. Cada unidad tiene cuatro compuertas (olvido, entrada, salida y valor candidato), cada una con pesos para las 52 entradas del día y para los 256 estados del día anterior. Con 256 unidades, el modelo tiene 317 697 parámetros:

| Parte | Cálculo | Parámetros |
|---|---|---|
| Pesos de las entradas | 4 compuertas × 256 unidades × 52 entradas | 53 248 |
| Pesos del estado del día anterior | 4 × 256 × 256 | 262 144 |
| Sesgos del LSTM | 2 × 4 × 256 | 2 048 |
| Capa final | 256 pesos + 1 sesgo | 257 |

Más unidades dan más capacidad para representar cuencas distintas, pero aumentan el riesgo de sobreajuste y el tiempo por época.

### 2.3 Entrenamiento

| Parámetro | Valor | Qué hace | Origen |
|---|---|---|---|
| `seq_length` | 365 | Días de clima que lee el modelo para predecir un día. Un año cubre las dos temporadas de lluvia y las dos secas; sin caudal rezagado, el modelo necesita esa historia para deducir cuánta agua guarda la cuenca | Chao et al. (2026); Wi et al.; Alzhanov et al. |
| `predict_last_n` | 1 | Solo el último día de cada ventana entra en la pérdida | |
| `batch_size` | 256 | Ventanas por actualización de pesos. El equipo pasó de 64 a 256 (11/11/2026) porque cada época es 2,8 veces más rápida (3.1). Si el flujo temporal no alcanza la meta, se prueba 64 | 64: Wi et al.; Alzhanov et al. 256: decisión nuestra |
| `epochs` | 30 | Máximo de pasadas por todas las ventanas | Propuesta |
| `optimizer`, `learning_rate` | Adam, 5 × 10⁻⁴ | Optimizador y tamaño del paso de cada actualización | Propuesta; Wi et al.; Alzhanov et al. |
| `loss` | `NSE` | NSE* de Kratzert: el error de cada día se divide por (σ_cuenca + 0,1)², para que las cuencas muy variables no dominen el entrenamiento | He et al. (2024); Yu & Tolson (2026) |
| `clip_gradient_norm` | 1 | Recorta los gradientes con norma mayor que 1, para evitar saltos bruscos de los pesos | Ejemplos de NeuralHydrology |
| `early_stopping` | `True`, paciencia 5, mínimo 5 épocas | Se detiene tras 5 épocas seguidas sin mejora de la pérdida de validación, contadas desde la época 6. Se evalúa con los pesos de la mejor época (sección 4) | Propuesta; paciencia y mínimo: decisión nuestra |
| `patience_dynamic_learning_rate`, `factor_dynamic_learning_rate` | 30, 0,5 | Desactivan en la práctica la reducción automática de la tasa de aprendizaje. Por un error de NeuralHydrology 1.13, `early_stopping: True` también activa esa reducción y exige estos dos valores. Con paciencia 30 (el máximo de épocas) nunca se aplica, y la tasa queda constante, como en la propuesta | Decisión nuestra |
| `seed` | 42 | Misma inicialización y mismo orden de lotes en cada corrida | Reproducibilidad |

### 2.4 Validación, registro y hardware

- Se valida al final de cada época (`validate_every: 1`, lo exige la parada temprana) en las cuencas de validación de cada etapa (195 en el flujo temporal).
- Métricas: NSE, KGE, RMSE, FHV, FLV y FMS; las tres últimas miden el sesgo en caudales altos, bajos y medios (Yilmaz et al., 2008). MAE no está en NeuralHydrology y se calcula aparte (sección 6).
- Se guarda un checkpoint por época y el registro de TensorBoard en `runs/`, que está excluida de git.
- Entrena en la GPU (`cuda:0`) con `num_workers: 0`, porque en Windows los valores mayores pueden bloquear el notebook.

### 2.5 Configuración en código

In [8]:
def fechas(periodo):
    return [pd.Timestamp(f).strftime("%d/%m/%Y") for f in PERIODOS[periodo]]

config_base = {
    # ---- datos y períodos (2.1)
    "experiment_name": "camels_col_lstm",
    "run_dir": "runs",
    "dataset": "generic",
    "data_dir": "data/processed",
    "train_basin_file": "configs/basins/todas.txt",
    "validation_basin_file": "configs/basins/validacion.txt",
    "test_basin_file": "configs/basins/prueba.txt",
    "train_start_date": fechas("train")[0],
    "train_end_date": fechas("train")[1],
    "validation_start_date": fechas("validation")[0],
    "validation_end_date": fechas("validation")[1],
    "test_start_date": fechas("test")[0],
    "test_end_date": fechas("test")[1],
    "dynamic_inputs": FORZANTES,
    "target_variables": [OBJETIVO],
    "static_attributes": ATRIBUTOS,
    "clip_targets_to_zero": [OBJETIVO],
    # ---- arquitectura (2.2)
    "model": "cudalstm",
    "hidden_size": 256,
    "initial_forget_bias": 3,
    "output_dropout": 0.4,
    "head": "regression",
    "output_activation": "linear",
    # ---- entrenamiento (2.3)
    "seq_length": LARGO_VENTANA,
    "predict_last_n": 1,
    "batch_size": 256,   # 64 en la propuesta; 256 desde el 11/11/2026 (3.1)
    "epochs": 30,
    "optimizer": "Adam",
    "learning_rate": 5.0e-4,
    "loss": "NSE",
    "clip_gradient_norm": 1,
    "early_stopping": True,
    "patience_early_stopping": 5,
    "minimum_epochs_before_early_stopping": 5,
    "patience_dynamic_learning_rate": 30,   # neutraliza la reducción de la tasa de aprendizaje (2.3)
    "factor_dynamic_learning_rate": 0.5,
    "seed": 42,
    # ---- validación, registro y hardware (2.4)
    "validate_every": 1,
    "validate_n_random_basins": len(validan),
    "metrics": ["NSE", "KGE", "RMSE", "FHV", "FLV", "FMS"],
    "save_weights_every": 1,
    "save_validation_results": False,
    "log_interval": 10,
    "log_tensorboard": True,
    "log_n_figures": 0,
    "device": "cuda:0",
    "num_workers": 0,
    "cache_validation_data": True,
}

def escribir_config(config, ruta):
    encabezado = "# Generado por proyecto-final-modelado.ipynb. Para cambiar un valor, edítalo en el notebook y vuelve a ejecutarlo.\n"
    ruta.write_text(encabezado + yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

escribir_config(config_base, ROOT / "configs" / "base.yml")
print((ROOT / "configs" / "base.yml").read_text(encoding="utf-8")[:1500], "...")

# Generado por proyecto-final-modelado.ipynb. Para cambiar un valor, edítalo en el notebook y vuelve a ejecutarlo.
experiment_name: camels_col_lstm
run_dir: runs
dataset: generic
data_dir: data/processed
train_basin_file: configs/basins/todas.txt
validation_basin_file: configs/basins/validacion.txt
test_basin_file: configs/basins/prueba.txt
train_start_date: 01/01/1981
train_end_date: 31/12/2010
validation_start_date: 01/01/2011
validation_end_date: 31/12/2015
test_start_date: 01/01/2016
test_end_date: 31/12/2022
dynamic_inputs:
- pr
- poten_evapo
- t_min
- t_max
target_variables:
- streamflow
static_attributes:
- log_area
- gravelius_index
- factor_form
- maximum_ele
- mean_ele
- minimum_ele
- streng_chanel
- equi_slope
- cn_catchment
- tc_Johnstone
- p_mean
- pet_mean
- aridity
- high_prec_freq
- high_prec_dur
- low_prec_freq
- low_prec_dur
- forest_perc
- nat_non_forest_form_perc
- agricul_livestock_perc
- non_vegeted_perc
- water_bodies_perc_cob
- unconso_depo_perc
- hypabyssal_roc

## 3. Prueba de humo

Antes del entrenamiento completo (varias horas) se corre uno corto para verificar toda la cadena: lectura de datos, entrenamiento, validación y checkpoints. La configuración es la misma de la sección 2 con estos cambios:

| Parámetro | Modelo principal | Prueba de humo | Por qué |
|---|---|---|---|
| `experiment_name` | `camels_col_lstm` | `camels_col_humo` | Separar las corridas |
| `train/validation/test_basin_file` | `todas.txt`, `validacion.txt`, `prueba.txt` | `humo_20.txt` (20) | 4 cuencas por región |
| `epochs` | 30 | 2 | Solo verificar que la pérdida baja |
| `validate_n_random_basins` | 195 | 20 | |
| `early_stopping` | `True` | `False` | Con 2 épocas no aplica |
| `log_n_figures` | 0 | 3 | Ver algunos hidrogramas |

In [9]:
config_humo = {
    **config_base,
    "experiment_name": "camels_col_humo",
    "train_basin_file": "configs/basins/humo_20.txt",
    "validation_basin_file": "configs/basins/humo_20.txt",
    "test_basin_file": "configs/basins/humo_20.txt",
    "epochs": 2,
    "validate_n_random_basins": 20,
    "early_stopping": False,
    "log_n_figures": 3,
}
escribir_config(config_humo, ROOT / "configs" / "humo.yml")
print("Escrito configs/humo.yml")

Escrito configs/humo.yml


In [10]:
EJECUTAR_PRUEBA_DE_HUMO = True

if EJECUTAR_PRUEBA_DE_HUMO:
    import torch
    from neuralhydrology.nh_run import start_run

    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")
    start_run(config_file=ROOT / "configs" / "humo.yml")
else:
    print("Prueba de humo desactivada: instala PyTorch y NeuralHydrology y cambia la bandera a True.")

GPU: NVIDIA GeForce RTX 2060 with Max-Q Design
2026-11-11 17:04:26,222: Logging to runs\camels_col_humo_1111_170426\output.log initialized.


2026-11-11 17:04:26,223: ### Folder structure created at runs\camels_col_humo_1111_170426


2026-11-11 17:04:26,224: ### Run configurations for camels_col_humo


2026-11-11 17:04:26,225: experiment_name: camels_col_humo


2026-11-11 17:04:26,226: run_dir: runs\camels_col_humo_1111_170426


2026-11-11 17:04:26,227: dataset: generic


... (2,072 outputs left out)


# Validation:  95%|█████████▌| 19/20 [00:06<00:00,  2.98it/s]


# Validation: 100%|██████████| 20/20 [00:06<00:00,  2.95it/s]


# Validation: 100%|██████████| 20/20 [00:06<00:00,  2.89it/s]

2026-11-11 17:06:47,653: Epoch 2 average validation loss: 0.04493 -- Median validation metrics: avg_loss: 0.04493, NSE: 0.38559, KGE: 0.50841, RMSE: 1.84143, FHV: -34.50566, FLV: -1.96353, FMS: -14.79453


**Qué revisar después de la prueba de humo:**

1. La pérdida de entrenamiento baja entre la época 1 y la 2.
2. Se reportan métricas de validación (NSE, KGE…) sin errores.
3. Existe `runs/camels_col_humo_<fecha>/` con `model_epoch001.pt` y `model_epoch002.pt`.
4. **Tiempo por época:** escalarlo de 20 a 248 cuencas para estimar el entrenamiento completo.
5. **Tamaño de lote:** comparar el tiempo con `batch_size: 64` y `256` (3.1).

**Resultado con `batch_size: 64`** (10/11/2026, `runs/camels_col_humo_1011_165655`): la pérdida de entrenamiento bajó de 0,052 a 0,041; la validación reportó NSE 0,46, KGE 0,48 y RMSE 1,79 mm/día (medianas de las 20 cuencas, época 2), y se guardaron los dos checkpoints. Cada época tomó unos 2,5 minutos. Las 248 cuencas tienen 14 veces más días de entrenamiento, así que con `batch_size: 64` se esperan unos 35 minutos por época. El punto 5 está en 3.1.

**Resultado con `batch_size: 256`** (11/11/2026, `runs/camels_col_humo_1111_170426`, la salida de arriba): la pérdida de entrenamiento bajó de 0,059 a 0,043 y la de validación de 0,046 a 0,045. La validación reportó NSE 0,39, KGE 0,51 y RMSE 1,84 mm/día (medianas, época 2), y se guardaron los dos checkpoints. Cada época tomó entre 52 y 58 s de entrenamiento y entre 8 y 15 s de validación, así que con las 248 cuencas se esperan unos 14 minutos por época. En los hidrogramas de validación (`img_log/`), la simulación sigue las dos temporadas de lluvia y las recesiones, pero subestima los picos (FHV −35 %), lo esperable tras solo 2 épocas.

**Reproducibilidad:** la prueba con lote 256 dio exactamente los mismos valores que la de 3.1, y una repetición con lote 64 (`runs/camels_col_humo_1111_164224`) dio los mismos que la del 10/11. Con `seed: 42`, repetir una corrida da el mismo resultado.

### 3.1 Tamaño de lote

Con `batch_size: 64`, una época de las 248 cuencas son 28 425 actualizaciones de pesos (1 819 212 ventanas de entrenamiento). La prueba de humo se corrió con 64 y con 256, sin cambiar nada más, para medir cuánto se acelera cada época.

La prueba mide velocidad, no calidad: con 4 veces menos actualizaciones por época, la pérdida puede bajar más despacio por época. La calidad solo se compara con entrenamientos completos.

La celda siguiente repite la prueba con `batch_size: 64`, por si hace falta comparar de nuevo.

In [10]:
config_humo_64 = {**config_humo, "experiment_name": "camels_col_humo_b64", "batch_size": 64}
escribir_config(config_humo_64, ROOT / "configs" / "humo_b64.yml")

EJECUTAR_PRUEBA_LOTE_64 = False   # ya se corrió, con el nombre camels_col_humo: runs/camels_col_humo_1011_165655

if EJECUTAR_PRUEBA_LOTE_64:
    from neuralhydrology.nh_run import start_run
    start_run(config_file=ROOT / "configs" / "humo_b64.yml")

**Resultado** (11/11/2026, `runs/camels_col_humo_b256_1111_135228`, comparado con `runs/camels_col_humo_1011_165655`):

| 20 cuencas, 2 épocas | `batch_size: 64` | `batch_size: 256` |
|---|---|---|
| Actualizaciones por época | 1981 | 496 |
| Entrenamiento por época | 2 min 18 s a 2 min 26 s | 51 s |
| Validación por época | 13 a 20 s | 6 a 11 s |
| Pérdida de validación, épocas 1 y 2 | 0,044 y 0,046 | 0,046 y 0,045 |
| NSE mediano de validación, épocas 1 y 2 | 0,47 y 0,46 | 0,41 y 0,39 |
| KGE mediano de validación, épocas 1 y 2 | 0,49 y 0,48 | 0,42 y 0,51 |

Con lote 256 cada época es 2,8 veces más rápida. Tras dos épocas la pérdida de validación es parecida, pero el NSE mediano es menor: con 4 veces menos actualizaciones, el modelo avanza menos en cada época. La parada temprana lo compensa entrenando más épocas, y aun así el total es mucho menor (4.1). **Decisión** (11/11/2026): todas las etapas usan `batch_size: 256`. Se comprobó con el flujo temporal completo (8.1): con lote 64 el NSE mediano de prueba fue el mismo, así que se mantiene 256.

## 4. Estrategia de modelado

El modelo es un LSTM regional (uno solo para todas las cuencas) entrenado desde cero con CAMELS-COL. Responde dos preguntas, cada una con su propio flujo de entrenamiento y evaluación:

| | Flujo temporal: cuenca vista, años nuevos | Flujo espacial: cuenca no vista |
|---|---|---|
| Pregunta | ¿Simula bien el caudal de una cuenca aforada en años que no vio? | ¿Simula bien el caudal de una cuenca sin estación de aforo? |
| Entrena con | Las 248 cuencas, 1981–2010 | Las 229 cuencas fuera de la partición *k*, 1981–2010 |
| Parada temprana con | Las 195 con al menos un año de datos en 2011–2015 | Las de entrenamiento con al menos un año de datos en 2011–2015 (177 a 184) |
| Se evalúa en | Las 183 con al menos un año de datos en 2016–2022 | Las 19 cuencas de la partición *k*, en todo su registro |
| Corridas | 1 | 5, una por partición |
| Meta | NSE y KGE medianos ≥ 0,70 | NSE mediano ≥ 0,50 |

Como el modelo nunca vio las cuencas retenidas, en el flujo espacial se evalúan en todo su registro (1981–2022), como dice la propuesta. También se reportan en 2016–2022, para compararlas con el flujo temporal en los mismos años.

**Etapas, en orden:**

1. **Ajuste de `hidden_size` (sección 7, opcional):** el equipo fijó 256 unidades (11/11/2026), el valor de la propuesta. Si alcanza el tiempo, se prueban 64, 128 y 256 solo con las 153 cuencas fijas, las que nunca se retienen en el flujo espacial, para que ninguna cuenca que luego hará de "no aforada" influya en la arquitectura.
2. **Flujo temporal (sección 8):** una corrida con `hidden_size` y `batch_size` de 256.
3. **Flujo espacial (sección 9):** cinco corridas con la misma configuración.
4. **Baselines y resultados (secciones 10 y 11):** referencias simples para juzgar las métricas, y comparación con las metas.

**Parada temprana y mejor época.**
- Al final de cada época se calcula la pérdida en el período de validación (2011–2015) de las cuencas de entrenamiento.
- El entrenamiento se detiene tras 5 épocas sin mejora, contadas desde la época 6. Cada corrida dura entre 11 y 30 épocas.
- NeuralHydrology conserva los pesos de la última época, no los de la mejor. Por eso siempre se evalúa con la época de menor pérdida de validación (`mejor_epoca`, sección 6).
- El período de prueba (2016–2022) no interviene en ninguna decisión: ni en la parada, ni en la elección de la época, ni en `hidden_size`.

**Cómo se evita la fuga de información.**
- **Anidamiento:** en el flujo espacial, una cuenca retenida sale del entrenamiento junto con todas las conectadas a ella aguas arriba o aguas abajo (sección 5). La sección 5.4 verifica que ningún par anidado quede repartido entre entrenamiento y prueba.
- **Estandarización:** NeuralHydrology calcula medias y desviaciones en cada corrida, solo con sus cuencas y su período de entrenamiento.
- **Entradas:** el modelo nunca recibe caudal.

**Interrupciones.** Cada época guarda un checkpoint. Si un entrenamiento se corta (apagón, reinicio), basta con volver a ejecutar su celda: `entrenar` retoma la corrida desde el último checkpoint. Las corridas terminadas no se repiten.

### 4.1 Tiempo de cómputo

Estimación con las velocidades medidas en las pruebas de humo (3.1) y las ventanas de entrenamiento de cada etapa. Cada corrida dura entre 11 y 30 épocas, según cuándo se active la parada temprana.

| Etapa | Corridas | Ventanas de entrenamiento por corrida | Minutos por época, lote 64 | Minutos por época, lote 256 | Horas en total, lote 64 | Horas en total, lote 256 |
|---|---|---|---|---|---|---|
| 1. Ajuste de `hidden_size` | 3 | 1,22 millones (153 cuencas) | 24 | 9 | 13 a 36 | 5 a 14 |
| 2. Flujo temporal | 1 | 1,82 millones (248 cuencas) | 36 | 14 | 7 a 18 | 3 a 7 |
| 3. Flujo espacial | 5 | 1,70 millones (229 cuencas) | 34 | 13 | 31 a 85 | 12 a 32 |
| **Total** | **9** | | | | **51 a 139** | **19 a 53** |

- Con lote 64, el plan completo puede tomar casi 6 días de GPU continua; con lote 256, entre 1 y 2 días.
- **Plan vigente** (11/11/2026): lote 256 y `hidden_size` fijo en 256, sin la etapa 1. Son 6 corridas, entre 15 y 39 horas; el flujo temporal va primero (3 a 7 horas).
- Las corridas son independientes entre sí, porque `hidden_size` quedó fijo (sección 7). Se pueden repartir entre computadores (por ejemplo, las 5 particiones), con el mismo código y la misma base, copiando después las carpetas de `runs/` a un solo lugar.

## 5. Anidamiento y particiones espaciales

En el flujo espacial, las cuencas de prueba hacen de cuencas sin aforo: el modelo no debe haber visto su caudal. Pero la mayoría de las cuencas están **anidadas**: una contiene a otra aguas arriba. El caudal de la cuenca de arriba es parte del caudal de la de abajo, así que entrenar con una y evaluar en la otra filtra información.

Por eso las particiones se forman con **sistemas de ríos completos**: grupos de cuencas conectadas por anidamiento, que siempre quedan juntos en entrenamiento o juntos en prueba (propuesta, sección 6.2). El anidamiento no viene en el dataset; se calcula con los polígonos de cuenca y los puntos de aforo de `03_CAMELS_COL_Basin_boundary`.

### 5.1 Polígonos y puntos de aforo

Los shapefiles se leen con un lector mínimo (`leer_shp`), para no agregar librerías geoespaciales al entorno. Están en la proyección World Mercator, en metros. Como control, el área de cada polígono se compara con el atributo `area` del dataset.

In [11]:
DIR_LIMITES = ROOT / "data" / "raw" / "03_CAMELS_COL_Basin_boundary" / "03_CAMELS_COL_Basin_boundary"
assert DIR_LIMITES.exists(), "Descomprime data/raw/03_CAMELS_COL_Basin_boundary.zip"


def leer_shp(ruta):
    """Lector mínimo de shapefile. Puntos (tipo 1): arreglo (x, y). Polígonos (tipo 5): lista de anillos (n x 2)."""
    b = Path(ruta).read_bytes()
    formas, pos = [], 100                                       # el encabezado del archivo ocupa 100 bytes
    while pos < len(b):
        largo = int.from_bytes(b[pos + 4:pos + 8], "big") * 2   # largo del registro, en palabras de 16 bits
        registro = b[pos + 8:pos + 8 + largo]
        tipo = int.from_bytes(registro[:4], "little")
        if tipo == 1:
            formas.append(np.frombuffer(registro, "<f8", 2, 4))
        elif tipo == 5:
            n_anillos, n_puntos = np.frombuffer(registro, "<i4", 2, 36)
            inicios = np.frombuffer(registro, "<i4", n_anillos, 44)
            xy = np.frombuffer(registro, "<f8", 2 * n_puntos, 44 + 4 * n_anillos).reshape(-1, 2)
            formas.append(np.split(xy, inicios[1:]))
        else:
            raise ValueError(f"Tipo de forma no soportado: {tipo}")
        pos += 8 + largo
    return formas


def leer_ids_dbf(ruta):
    """Identificadores de la tabla .dbf del shapefile (tiene un solo campo, IDEAM_CODE)."""
    b = Path(ruta).read_bytes()
    n, largo_encabezado, largo_registro = struct.unpack("<IHH", b[4:12])
    assert largo_encabezado == 65, "Se esperaba un solo campo en la tabla"
    return [b[largo_encabezado + i * largo_registro + 1:largo_encabezado + (i + 1) * largo_registro].decode().strip()
            for i in range(n)]


poligonos = dict(zip(leer_ids_dbf(DIR_LIMITES / "CAMELS_COL_catchments_boundaries.dbf"),
                     leer_shp(DIR_LIMITES / "CAMELS_COL_catchments_boundaries.shp")))
aforos = dict(zip(leer_ids_dbf(DIR_LIMITES / "CAMELS_COL_GAUGING.dbf"),
                  leer_shp(DIR_LIMITES / "CAMELS_COL_GAUGING.shp")))
assert set(cuencas) <= set(poligonos) and set(cuencas) <= set(aforos), "Faltan cuencas en los shapefiles"

# Control de áreas. En Mercator las distancias se estiran por 1/cos(lat): el área se corrige por cos²(lat).
R_TIERRA = 6378137.0


def area_km2(anillos):
    xy = np.vstack(anillos)
    lat = 2 * np.arctan(np.exp(xy[:, 1].mean() / R_TIERRA)) - np.pi / 2
    gauss = sum(abs(np.dot(a[:-1, 0], a[1:, 1]) - np.dot(a[1:, 0], a[:-1, 1])) / 2 for a in anillos)
    return gauss * np.cos(lat) ** 2 / 1e6


area = pd.read_csv(DIR_BASE / "atributos.csv", dtype={"gauge_id": str}, index_col="gauge_id").loc[cuencas, "area"]
razon = pd.Series({g: area_km2(poligonos[g]) for g in cuencas}) / area
print(f"{len(poligonos)} polígonos y {len(aforos)} aforos en los shapefiles; se usan las {len(cuencas)} cuencas del modelo")
print(f"Área del polígono / atributo `area`: {razon.min():.3f} a {razon.max():.3f}")

346 polígonos y 346 aforos en los shapefiles; se usan las 248 cuencas del modelo
Área del polígono / atributo `area`: 0.964 a 1.000


### 5.2 Anidamiento

La cuenca *i* está anidada en la cuenca *j* si el punto de aforo de *i* cae dentro del polígono de *j* (propuesta, 6.2): todo lo que drena por el aforo de *i* sigue río abajo y pasa por el de *j*.

Se verifica el criterio con las áreas, usando una rejilla de puntos dentro de cada polígono:
- En los pares anidados, la cuenca de arriba debe quedar entera dentro de la de abajo y ser más pequeña.
- Los pares que no están anidados no deben compartir área. Por errores de digitalización los bordes de cuencas vecinas se tocan, así que se admite una superposición pequeña.

La celda tarda cerca de un minuto.

In [12]:
trazos = {g: Trazo.make_compound_path(*[Trazo(a) for a in poligonos[g]]) for g in cuencas}
cajas = {g: np.r_[np.vstack(poligonos[g]).min(0), np.vstack(poligonos[g]).max(0)] for g in cuencas}
xy_aforos = np.array([aforos[g] for g in cuencas])

# dentro[i, j]: el aforo de la cuenca i cae dentro del polígono de la cuenca j (i está aguas arriba de j)
n = len(cuencas)
dentro = np.zeros((n, n), bool)
for j, g in enumerate(cuencas):
    x0, y0, x1, y1 = cajas[g]
    candidatas = np.where((xy_aforos[:, 0] >= x0) & (xy_aforos[:, 0] <= x1) &
                          (xy_aforos[:, 1] >= y0) & (xy_aforos[:, 1] <= y1))[0]
    candidatas = candidatas[candidatas != j]
    dentro[candidatas, j] = trazos[g].contains_points(xy_aforos[candidatas])


def puntos_internos(g, lado=12):
    """Puntos de una rejilla de lado x lado sobre la caja del polígono que caen dentro de él."""
    x0, y0, x1, y1 = cajas[g]
    xx, yy = np.meshgrid(np.linspace(x0, x1, lado + 2)[1:-1], np.linspace(y0, y1, lado + 2)[1:-1])
    p = np.c_[xx.ravel(), yy.ravel()]
    return p[trazos[g].contains_points(p)]


def se_cruzan(a, b):
    return not (a[0] > b[2] or a[2] < b[0] or a[1] > b[3] or a[3] < b[1])


muestras = {g: puntos_internos(g) for g in cuencas}
arriba, abajo = np.where(dentro)
contenida = np.array([trazos[cuencas[j]].contains_points(muestras[cuencas[i]]).mean() for i, j in zip(arriba, abajo)])
compartida = np.array([trazos[cuencas[j]].contains_points(muestras[cuencas[i]]).mean()
                       for i in range(n) for j in range(n)
                       if i != j and not (dentro[i, j] or dentro[j, i]) and se_cruzan(cajas[cuencas[i]], cajas[cuencas[j]])])

assert (area.to_numpy()[arriba] < area.to_numpy()[abajo]).all(), "Hay una cuenca de arriba más grande que la de abajo"
assert contenida.min() > 0.95 and compartida.max() < 0.05, "El anidamiento por aforos no coincide con las áreas"
print(f"Pares anidados: {dentro.sum()}. Cuencas anidadas con al menos otra: {(dentro | dentro.T).any(axis=1).sum()} de {n}")
print(f"  Área de la cuenca de arriba dentro de la de abajo: {contenida.min():.0%} como mínimo")
print(f"  Pares no anidados con cajas que se cruzan: {len(compartida)}; área compartida: {compartida.max():.1%} como máximo")

Pares anidados: 1044. Cuencas anidadas con al menos otra: 229 de 248
  Área de la cuenca de arriba dentro de la de abajo: 98% como mínimo
  Pares no anidados con cajas que se cruzan: 4320; área compartida: 2.5% como máximo


### 5.3 Sistemas de ríos

Un sistema de ríos es un grupo de cuencas conectadas por anidamiento, directa o indirectamente: las componentes conexas del grafo de 5.2. Los sistemas se numeran de mayor a menor.

In [13]:
n_sistemas, etiqueta = connected_components(dentro | dentro.T, directed=False)
cuenca_sistema = pd.DataFrame({"gauge_id": cuencas, "componente": etiqueta}).merge(resumen[["gauge_id", "region"]])

# numeración de mayor a menor (a igual tamaño, por región)
tamanos = (cuenca_sistema.groupby("componente")
           .agg(cuencas=("gauge_id", "size"), region=("region", lambda r: r.mode()[0]), regiones=("region", "nunique"))
           .sort_values(["cuencas", "region"], ascending=[False, True]))
tamanos["sistema"] = range(1, len(tamanos) + 1)
cuenca_sistema["sistema"] = cuenca_sistema["componente"].map(tamanos["sistema"])
sistemas = tamanos.set_index("sistema")
assert (sistemas["regiones"] == 1).all(), "Hay un sistema de ríos repartido en dos regiones"

print(f"{n_sistemas} sistemas de ríos; {(sistemas['cuencas'] == 1).sum()} son cuencas aisladas (sin otra anidada)")
print("Cuencas por sistema:", sistemas["cuencas"].value_counts().sort_index(ascending=False).to_dict())
sistemas.head(8)[["region", "cuencas"]]

35 sistemas de ríos; 19 son cuencas aisladas (sin otra anidada)
Cuencas por sistema: {130: 1, 23: 1, 14: 1, 10: 1, 9: 1, 7: 2, 5: 1, 4: 3, 3: 2, 2: 3, 1: 19}


,region,cuencas
sistema,,
1,Magdalena-Cauca,130
2,Orinoco,23
3,Pacific,14
4,Caribbean,10
5,Orinoco,9
6,Caribbean,7
7,Caribbean,7
8,Amazon,5


### 5.4 Particiones de la validación cruzada espacial

Como en la propuesta, los dos sistemas más grandes (Magdalena-Cauca y Orinoco) quedan **siempre en entrenamiento**: retener cualquiera dejaría sin datos a su región. Son las **cuencas fijas**, que se usan también para ajustar `hidden_size` (sección 7).

Los demás sistemas se reparten en 5 particiones, de mayor a menor: cada sistema va a la partición con menos cuencas y, si hay empate, a la que tiene menos cuencas de su región. Así las particiones quedan del mismo tamaño y con las regiones lo más mezcladas posible.

Para cada partición *k* se escriben tres listas: `particion_k_prueba.txt` (las retenidas), `particion_k_entrenamiento.txt` (todas las demás, fijas incluidas) y `particion_k_validacion.txt` (las de entrenamiento con al menos un año de datos en 2011–2015, 1.2). Para las cuencas fijas se escriben `fijas.txt` y `fijas_validacion.txt`. Antes se verifica:
- que ningún par anidado quede con una cuenca en entrenamiento y la otra en prueba;
- que ningún atributo estático sea constante entre las cuencas de entrenamiento, porque NeuralHydrology no entrena en ese caso (no puede estandarizarlo);
- que todas las cuencas retenidas tengan al menos un año de días evaluables en su registro.

In [14]:
N_PARTICIONES = 5
SISTEMAS_FIJOS = [1, 2]
assert sistemas.loc[SISTEMAS_FIJOS, "region"].tolist() == ["Magdalena-Cauca", "Orinoco"]

particion_de = {s: 0 for s in SISTEMAS_FIJOS}            # 0 = siempre en entrenamiento
cuentas = {k: 0 for k in range(1, N_PARTICIONES + 1)}     # cuencas por partición
cuentas_region = {}                                       # (partición, región) -> cuencas
for s, fila in sistemas.drop(SISTEMAS_FIJOS).iterrows():  # de mayor a menor
    k = min(cuentas, key=lambda k: (cuentas[k], cuentas_region.get((k, fila["region"]), 0), k))
    particion_de[s] = k
    cuentas[k] += fila["cuencas"]
    cuentas_region[(k, fila["region"])] = cuentas_region.get((k, fila["region"]), 0) + fila["cuencas"]
cuenca_sistema["particion"] = cuenca_sistema["sistema"].map(particion_de)

def escribir_lista(nombre, ids):
    (DIR_CUENCAS / nombre).write_text("\n".join(ids) + "\n")


posicion = {g: i for i, g in enumerate(cuencas)}
fijas = cuenca_sistema.loc[cuenca_sistema["particion"] == 0, "gauge_id"].tolist()
fijas_validacion = [g for g in fijas if g in set(validan)]
escribir_lista("fijas.txt", fijas)
escribir_lista("fijas_validacion.txt", fijas_validacion)
validan_por_particion = {}
for k in range(1, N_PARTICIONES + 1):
    prueba = cuenca_sistema.loc[cuenca_sistema["particion"] == k, "gauge_id"].tolist()
    entrenamiento = cuenca_sistema.loc[cuenca_sistema["particion"] != k, "gauge_id"].tolist()
    p, e = [posicion[g] for g in prueba], [posicion[g] for g in entrenamiento]
    assert not (dentro[np.ix_(p, e)].any() or dentro[np.ix_(e, p)].any()), f"Partición {k}: hay un par anidado repartido"
    constantes = atributos_modelo.loc[entrenamiento].std().pipe(lambda d: d[d == 0].index.tolist())
    assert not constantes, f"Partición {k}: atributos constantes en entrenamiento {constantes}"
    assert dias_evaluables.loc[prueba].sum(axis=1).min() >= MIN_DIAS_EVALUACION, f"Partición {k}: retenida sin datos"
    validan_por_particion[k] = [g for g in entrenamiento if g in set(validan)]
    escribir_lista(f"particion_{k}_prueba.txt", prueba)
    escribir_lista(f"particion_{k}_entrenamiento.txt", entrenamiento)
    escribir_lista(f"particion_{k}_validacion.txt", validan_por_particion[k])
assert not atributos_modelo.loc[fijas].std().eq(0).any(), "Atributos constantes en las cuencas fijas"
cuenca_sistema[["gauge_id", "region", "sistema", "particion"]].to_csv(DIR_CUENCAS / "particiones.csv", index=False)

etiqueta_particion = cuenca_sistema["particion"].map(lambda k: "fijas" if k == 0 else f"partición {k}")
anio_validacion = cuenca_sistema["gauge_id"].map(dias_evaluables["validation"]) >= MIN_DIAS_EVALUACION
anio_prueba = cuenca_sistema["gauge_id"].map(dias_evaluables["test"]) >= MIN_DIAS_EVALUACION
tabla = pd.crosstab(etiqueta_particion, cuenca_sistema["region"], margins=True, margins_name="Total")
tabla["sistemas"] = cuenca_sistema.groupby(etiqueta_particion)["sistema"].nunique()
tabla["≥ 1 año en 2011–2015"] = anio_validacion.groupby(etiqueta_particion).sum()
tabla["≥ 1 año en 2016–2022"] = anio_prueba.groupby(etiqueta_particion).sum()
tabla.loc["Total", ["sistemas", "≥ 1 año en 2011–2015", "≥ 1 año en 2016–2022"]] = [
    n_sistemas, anio_validacion.sum(), anio_prueba.sum()]
print("Escritas fijas.txt, fijas_validacion.txt, particion_k_{prueba,entrenamiento,validacion}.txt (k = 1 a 5) y particiones.csv")
print(f"Cuencas que validan: {len(fijas_validacion)} de las fijas; en las particiones, "
      f"{min(map(len, validan_por_particion.values()))} a {max(map(len, validan_por_particion.values()))}")
tabla.astype(int)

Escritas fijas.txt, fijas_validacion.txt, particion_k_{prueba,entrenamiento,validacion}.txt (k = 1 a 5) y particiones.csv
Cuencas que validan: 125 de las fijas; en las particiones, 177 a 184


region,Amazon,Caribbean,Magdalena-Cauca,Orinoco,Pacific,Total,sistemas,≥ 1 año en 2011–2015,≥ 1 año en 2016–2022
particion,,,,,,,,,
fijas,0,0,130,23,0,153,2,125,121
partición 1,0,2,2,1,14,19,5,15,12
partición 2,0,12,2,0,5,19,6,13,13
partición 3,1,4,1,13,0,19,7,18,16
partición 4,5,13,1,0,0,19,7,11,11
partición 5,5,13,0,0,1,19,8,13,10
Total,11,44,136,37,20,248,35,195,183


Cada partición retiene 19 cuencas y entrena con las otras 229. Las 95 cuencas evaluadas como no aforadas son todas las del Caribe, el Pacífico y el Amazonas, más 14 del Orinoco y 6 de Magdalena-Cauca. Como advierte la propuesta, son las regiones más atípicas y con menos cuencas de entrenamiento parecidas, así que la meta de NSE ≥ 0,50 en el flujo espacial es una cota inferior conservadora del desempeño en cuencas no aforadas.

## 6. Funciones de entrenamiento y evaluación

| Función | Qué hace |
|---|---|
| `entrenar(config)` | Escribe la configuración en `configs/corridas/` y entrena. Si la corrida ya terminó, la reutiliza; si quedó a medias, la retoma desde su último checkpoint |
| `corrida_terminada(config)` | Carpeta de la corrida terminada con esa configuración, o `None` |
| `historial(run_dir)` | Pérdidas de entrenamiento y validación y medianas de validación por época, leídas de `output.log` |
| `mejor_epoca(run_dir)` | Época con la menor pérdida de validación |
| `evaluar(run_dir)` | Simulación del período de prueba con los pesos de la mejor época: caudal observado y simulado por cuenca y día |
| `metricas(series, desde, hasta)` | NSE, KGE, RMSE, FHV, FLV, FMS y MAE por cuenca, entre dos fechas |
| `resumir(tabla)` | Medianas entre cuencas y número de cuencas con NSE negativo |

Detalles:
- **Una corrida por configuración.** El nombre de cada corrida lleva una huella de 6 caracteres calculada con toda su configuración. Si se cambia cualquier valor (por ejemplo, `batch_size`), se entrena una corrida nueva en vez de reutilizar la anterior.
- **Al retomar**, NeuralHydrology guarda los checkpoints nuevos en una subcarpeta `continue_training_from_epochNNN` y reinicia el contador de la parada temprana. `entrenar` copia esos checkpoints a la carpeta principal para que todos queden juntos.
- **Registro:** NeuralHydrology configura su registro una sola vez por kernel. `entrenar` lo reinicia antes de cada corrida; si no, desde la segunda corrida todo se escribiría en el `output.log` de la primera.
- **Métricas:** solo cuentan los días con caudal observado y simulado. Las predicciones negativas se llevan a 0, como en NeuralHydrology. Una cuenca se evalúa si tiene al menos 365 días válidos en el período (`MIN_DIAS_EVALUACION`, 1.2). Las medianas coinciden con las que reporta NeuralHydrology.

In [15]:
from neuralhydrology.evaluation.metrics import calculate_metrics
from neuralhydrology.nh_run import continue_run, eval_run
from neuralhydrology.training.train import start_training
from neuralhydrology.utils.config import Config

DIR_CONFIGS_CORRIDAS = ROOT / "configs" / "corridas"
DIR_CONFIGS_CORRIDAS.mkdir(exist_ok=True)
METRICAS_NH = config_base["metrics"]


# ---- carpetas de corrida

def nombre_corrida(config):
    """Nombre del experimento más una huella de 6 caracteres de toda la configuración."""
    resto = {k: v for k, v in config.items() if k != "experiment_name"}
    huella = hashlib.md5(yaml.safe_dump(resto, sort_keys=True).encode()).hexdigest()[:6]
    return f"{config['experiment_name']}_{huella}"


def corridas(config):
    """Carpetas de corrida de esta configuración, de la más reciente a la más antigua."""
    nombre = nombre_corrida(config)
    patron = re.compile(rf"{re.escape(nombre)}_\d{{4}}_\d{{6}}")
    carpetas = [p for p in (ROOT / config["run_dir"]).glob(f"{nombre}_*") if patron.fullmatch(p.name)]
    return sorted(carpetas, key=lambda p: p.stat().st_mtime, reverse=True)


def corrida_terminada(config):
    previas = corridas(config)
    return previas[0] if previas and (previas[0] / "terminado.txt").exists() else None


def reunir_checkpoints(run_dir):
    """Copia a la carpeta principal los checkpoints guardados al retomar y devuelve las épocas disponibles."""
    for archivo in run_dir.glob("continue_training_from_epoch*/*_epoch*.pt"):
        if not (run_dir / archivo.name).exists():
            shutil.copy2(archivo, run_dir / archivo.name)
    return sorted(int(p.stem[-3:]) for p in run_dir.glob("model_epoch*.pt"))


# ---- entrenamiento

def reiniciar_registro():
    """Cierra los archivos de registro abiertos, para que cada corrida escriba en su propio output.log."""
    raiz = logging.getLogger()
    for manejador in list(raiz.handlers):
        raiz.removeHandler(manejador)
        manejador.close()


def entrenar(config):
    """Entrena `config`, reutiliza su corrida terminada o retoma la interrumpida. Devuelve la carpeta de la corrida."""
    nombre = nombre_corrida(config)
    ruta = DIR_CONFIGS_CORRIDAS / f"{nombre}.yml"
    escribir_config({**config, "experiment_name": nombre}, ruta)

    terminada = corrida_terminada(config)
    if terminada is not None:
        print(f"{nombre}: ya entrenada ({terminada.name})")
        return terminada

    previas = corridas(config)
    epocas_previas = reunir_checkpoints(previas[0]) if previas else []
    if epocas_previas:
        run_dir, ultima = previas[0], epocas_previas[-1]
        faltan = config["epochs"] - ultima
        print(f"{nombre}: se retoma {run_dir.name} desde la época {ultima} ({faltan} por entrenar)")
        if faltan > 0:
            reiniciar_registro()
            continue_run(run_dir, config_file={"epochs": faltan})
            reunir_checkpoints(run_dir)
    else:
        reiniciar_registro()
        cfg = Config(ruta)
        start_training(cfg)
        run_dir = Path(cfg.run_dir)
    reiniciar_registro()

    (run_dir / "terminado.txt").write_text(f"Entrenamiento terminado el {datetime.now():%Y-%m-%d %H:%M}\n")
    return run_dir


# ---- historial y mejor época

PATRON_ENTRENAMIENTO = re.compile(r"Epoch (\d+) average loss: avg_loss: (\S+),")
PATRON_VALIDACION = re.compile(r"Epoch (\d+) average validation loss: (\S+?)(?: -- Median validation metrics: (.*))?$")


def historial(run_dir):
    """Pérdidas y medianas de validación por época. Al retomar, cada tramo tiene su output.log;
    si una época aparece en dos, vale el registro más reciente."""
    filas = {}
    for log in sorted(Path(run_dir).rglob("output.log"), key=lambda p: p.stat().st_mtime):
        for linea in log.read_text(encoding="utf-8", errors="ignore").splitlines():
            if m := PATRON_ENTRENAMIENTO.search(linea):
                filas.setdefault(int(m[1]), {})["perdida_entrenamiento"] = float(m[2])
            elif m := PATRON_VALIDACION.search(linea):
                fila = filas.setdefault(int(m[1]), {})
                fila["perdida_validacion"] = float(m[2])
                for par in (m[3] or "").split(", "):
                    if ": " in par and not par.startswith("avg_"):
                        clave, valor = par.split(": ")
                        fila[f"{clave}_validacion"] = float(valor)
    return pd.DataFrame.from_dict(filas, orient="index").rename_axis("epoca").sort_index()


def mejor_epoca(run_dir):
    return int(historial(run_dir)["perdida_validacion"].idxmin())


# ---- evaluación

def evaluar(run_dir, epoca=None):
    """Simulación del período de prueba de la corrida con los pesos de `epoca` (por defecto, la mejor).
    Devuelve {cuenca: DataFrame con columnas obs y sim, una fila por día}."""
    epoca = mejor_epoca(run_dir) if epoca is None else epoca
    archivo = Path(run_dir) / "test" / f"model_epoch{epoca:03d}" / "test_results.p"
    if not archivo.exists():
        eval_run(Path(run_dir), period="test", epoch=epoca)
    with archivo.open("rb") as f:
        resultados = pickle.load(f)
    series = {}
    for cuenca, r in resultados.items():
        ds = r["1D"]["xr"].isel(time_step=-1)
        series[cuenca] = pd.DataFrame({"obs": ds[f"{OBJETIVO}_obs"].to_series(),
                                       "sim": ds[f"{OBJETIVO}_sim"].to_series().clip(lower=0)})
    return series


def metricas(series, desde=None, hasta=None):
    """Métricas por cuenca entre `desde` y `hasta`, con los días que tienen caudal observado y simulado."""
    filas = {}
    for cuenca, df in series.items():
        df = df.loc[desde:hasta]
        validos = df["obs"].notna() & df["sim"].notna()
        if validos.sum() < MIN_DIAS_EVALUACION:
            continue
        obs, sim = (xr.DataArray(df[c].to_numpy(), coords={"date": df.index.to_numpy()}, dims="date")
                    for c in ["obs", "sim"])
        fila = calculate_metrics(obs, sim, metrics=METRICAS_NH, resolution="1D")
        fila["MAE"] = float((df["obs"] - df["sim"])[validos].abs().mean())
        fila["dias"] = int(validos.sum())
        filas[cuenca] = fila
    return pd.DataFrame.from_dict(filas, orient="index").rename_axis("gauge_id")


def resumir(tabla_metricas):
    """Medianas entre cuencas: el NSE no tiene cota inferior y una sola cuenca muy mala arruina el promedio."""
    return pd.Series({"cuencas": len(tabla_metricas),
                      **tabla_metricas[["NSE", "KGE", "RMSE", "MAE", "FHV", "FLV", "FMS"]].median().round(3),
                      "cuencas con NSE < 0": int((tabla_metricas["NSE"] < 0).sum())})

## 7. Etapa 1 (opcional): ajuste de `hidden_size`

**Decisión del equipo (11/11/2026):** `hidden_size = 256`, el valor de la propuesta, tomado de estudios con más cuencas. Las etapas 2 y 3 usan ese valor, y esta etapa queda para el final, si alcanza el tiempo.

Si se corre: la complejidad del LSTM debe corresponder al número de cuencas de entrenamiento (He et al., 2024), así que se prueban 64, 128 y 256 unidades.

- **Con qué:** las 153 cuencas fijas (5.4). Se entrena con las 153 en 1981–2010 y se valida en 2011–2015 con las 125 que tienen al menos un año de datos en ese período. Las cuencas de las particiones no intervienen.
- **Criterio:** el mayor NSE mediano de validación en la mejor época de cada corrida.
- **Resultado:** la tabla del ajuste. Si gana otro tamaño, el equipo decide si cambia `hidden_size` en `config_base` (2.5); eso crea corridas nuevas de las etapas 2 y 3.

Para entrenar, cambia `EJECUTAR_AJUSTE` a `True`. El tiempo estimado está en 4.1.

In [16]:
EJECUTAR_AJUSTE = False   # opcional: el equipo fijó hidden_size = 256 (11/11/2026)
TAMANOS_OCULTOS = [64, 128, 256]
HIDDEN_SIZE = config_base["hidden_size"]   # el que usan las etapas 2 y 3


def config_ajuste(h):
    return {**config_base, "experiment_name": f"camels_col_ajuste_h{h}", "hidden_size": h,
            "train_basin_file": "configs/basins/fijas.txt",
            "validation_basin_file": "configs/basins/fijas_validacion.txt",
            "test_basin_file": "configs/basins/fijas_validacion.txt",   # no se usa: la etapa solo mira la validación
            "validate_n_random_basins": len(fijas_validacion)}


filas = {}
for h in TAMANOS_OCULTOS:
    run_dir = entrenar(config_ajuste(h)) if EJECUTAR_AJUSTE else corrida_terminada(config_ajuste(h))
    if run_dir is not None:
        hist, mejor = historial(run_dir), mejor_epoca(run_dir)
        filas[h] = {"corrida": run_dir.name, "épocas": len(hist), "mejor época": mejor,
                    **hist.loc[mejor, ["perdida_validacion", "NSE_validacion", "KGE_validacion"]].round(3)}
ajuste = pd.DataFrame.from_dict(filas, orient="index").rename_axis("hidden_size")

print(f"hidden_size de las etapas 2 y 3: {HIDDEN_SIZE}")
if len(ajuste) == len(TAMANOS_OCULTOS):
    print(f"Mejor tamaño en el ajuste: {int(ajuste['NSE_validacion'].idxmax())}")
if len(ajuste):
    display(ajuste)
else:
    print("Ajuste no ejecutado (opcional)")

hidden_size de las etapas 2 y 3: 256
Ajuste no ejecutado (opcional)


## 8. Etapa 2: flujo temporal (cuenca vista)

El modelo principal se entrena con las 248 cuencas, con `hidden_size` y `batch_size` de 256. Se evalúa en 2016–2022 en las 183 cuencas con al menos un año de días evaluables en ese período (1.2).

Para entrenar, cambia `EJECUTAR_FLUJO_TEMPORAL` a `True`; toma entre 3 y 7 horas (4.1). El avance de cada época queda en `runs/camels_col_temporal_h256_<huella>_<fecha>/output.log`. Con la bandera en `False`, la celda usa la corrida si ya terminó.

In [17]:
EJECUTAR_FLUJO_TEMPORAL = False

config_temporal = {**config_base, "experiment_name": f"camels_col_temporal_h{HIDDEN_SIZE}", "hidden_size": HIDDEN_SIZE}
run_temporal = entrenar(config_temporal) if EJECUTAR_FLUJO_TEMPORAL else corrida_terminada(config_temporal)

series_temporal = evaluar(run_temporal) if run_temporal is not None else {}
if series_temporal:
    print(f"{run_temporal.name}: {len(historial(run_temporal))} épocas, mejor época {mejor_epoca(run_temporal)}")
    metricas_temporal = metricas(series_temporal)
    display(resumir(metricas_temporal))
else:
    print("Flujo temporal pendiente")

camels_col_temporal_h256_eaa981_1111_171248: 14 épocas, mejor época 3

cuencas                183.000
NSE                      0.394
KGE                      0.513
RMSE                     1.688
MAE                      1.063
FHV                    -29.752
FLV                      8.830
FMS                    -10.311
cuencas con NSE < 0     20.000
dtype: float64

### 8.1 Comparación: flujo temporal con lote 64

Con lote 256 (`runs/camels_col_temporal_h256_eaa981_1111_171248`, 11/11/2026), la parada temprana detuvo el entrenamiento en la época 14; la mejor fue la 3. En prueba, el NSE mediano fue 0,39 y el KGE 0,51, por debajo de la meta de 0,70. Como se acordó en 3.1, se repite con `batch_size: 64`, el valor de la propuesta, sin cambiar nada más. La comparación usa las mismas cuencas y los mismos días. Si el lote 64 mejora, el flujo espacial (sección 9) usa 64.

Para entrenar, cambia `EJECUTAR_FLUJO_TEMPORAL_LOTE_64` a `True`.

**Resultado** (12/11/2026, `runs/camels_col_temporal_h256_b64_ffa963_1111_222122`): la parada temprana detuvo el entrenamiento en la época 11, y la mejor también fue la 3. En prueba, el NSE mediano fue el mismo (0,394) y el KGE casi igual (0,511 frente a 0,513), con 23 cuencas con NSE negativo frente a 20. Cada época tomó unos 25 minutos (23 de entrenamiento y 2 de validación), menos que los 36 que estimaba 4.1.

**Decisión:** el tamaño de lote no cambia el desempeño, así que el flujo espacial usa lote 256, que es 1,5 veces más rápido (unos 17 minutos por época).

In [18]:
EJECUTAR_FLUJO_TEMPORAL_LOTE_64 = False

config_temporal_64 = {**config_temporal, "experiment_name": f"camels_col_temporal_h{HIDDEN_SIZE}_b64", "batch_size": 64}
run_temporal_64 = entrenar(config_temporal_64) if EJECUTAR_FLUJO_TEMPORAL_LOTE_64 else corrida_terminada(config_temporal_64)

series_temporal_64 = evaluar(run_temporal_64) if run_temporal_64 is not None else {}
if series_temporal_64:
    print(f"{run_temporal_64.name}: {len(historial(run_temporal_64))} épocas, mejor época {mejor_epoca(run_temporal_64)}")
    metricas_temporal_64 = metricas(series_temporal_64)
    comparacion_lote = {"lote 64": resumir(metricas_temporal_64)}
    if series_temporal:
        comparacion_lote = {"lote 256": resumir(metricas_temporal), **comparacion_lote}
    display(pd.DataFrame(comparacion_lote))
else:
    print("Flujo temporal con lote 64 pendiente")

camels_col_temporal_h256_b64_ffa963_1111_222122: 11 épocas, mejor época 3


,lote 256,lote 64
cuencas,183.000,183.000
NSE,0.394,0.394
KGE,0.513,0.511
RMSE,1.688,1.744
MAE,1.063,1.061
FHV,-29.752,-29.370
FLV,8.830,8.009
FMS,-10.311,-9.135
cuencas con NSE < 0,20.000,23.000


### 8.2 Modelo final: entrenamiento con 1981–2015

En las dos corridas anteriores (8 y 8.1), la mejor pérdida de validación llegó en la época 3. Además, buena parte del error en prueba viene de cuencas cuya relación entre lluvia y caudal cambió con los años. Por eso el modelo final se entrena también con los años de validación (2011–2015), los más cercanos a la prueba, durante 3 épocas fijas.

- **Sin fuga de información:** el número de épocas se eligió con la validación de 8 y 8.1. La prueba (2016–2022) sigue sin intervenir en ninguna decisión.
- **Sin validación ni parada temprana:** 2011–2015 ya es parte del entrenamiento, así que no queda período para validar (`validate_every: None`, `early_stopping: False`). Sin parada temprana, la tasa de aprendizaje queda constante (2.3).
- **Comparación:** con la corrida de 8 (lote 256, entrenada con 1981–2010, época 3), en las mismas cuencas y días de prueba.

Toma unos 50 minutos (3 épocas de unos 16 minutos). Para entrenar, cambia `EJECUTAR_MODELO_FINAL_TEMPORAL` a `True`.

**Resultado** (12/11/2026, `runs/camels_col_temporal_h256_1981_2015_129122_1211_224752`): el NSE mediano de prueba subió apenas de 0,394 a 0,398, el KGE quedó igual (0,513) y 20 cuencas siguen con NSE negativo. Solo la mitad de las cuencas mejora (90 de 183), y el error de volumen de las cuencas cuya relación entre lluvia y caudal cambió no se reduce. La mejora es marginal: sumar años recientes no basta. El siguiente paso es mejorar las entradas (sección 13).

In [19]:
EJECUTAR_MODELO_FINAL_TEMPORAL = False
EPOCAS_FINAL = 3   # la mejor época de validación con lote 256 y con lote 64 (8 y 8.1)

config_temporal_final = {**config_temporal, "experiment_name": f"camels_col_temporal_h{HIDDEN_SIZE}_1981_2015",
                         "train_end_date": fechas("validation")[1],   # entrena también con 2011–2015
                         "epochs": EPOCAS_FINAL, "early_stopping": False, "validate_every": None}
run_final = entrenar(config_temporal_final) if EJECUTAR_MODELO_FINAL_TEMPORAL else corrida_terminada(config_temporal_final)

series_final = evaluar(run_final, epoca=EPOCAS_FINAL) if run_final is not None else {}
if series_final:
    metricas_final = metricas(series_final)
    comparacion_final = {f"1981–{pd.Timestamp(PERIODOS['train'][1]).year}, época {mejor_epoca(run_temporal)}": resumir(metricas_temporal)} if series_temporal else {}
    comparacion_final[f"1981–2015, {EPOCAS_FINAL} épocas"] = resumir(metricas_final)
    display(pd.DataFrame(comparacion_final))
else:
    print("Modelo final del flujo temporal pendiente")

,"1981–2010, época 3","1981–2015, 3 épocas"
cuencas,183.000,183.000
NSE,0.394,0.398
KGE,0.513,0.513
RMSE,1.688,1.671
MAE,1.063,1.072
FHV,-29.752,-28.756
FLV,8.830,28.695
FMS,-10.311,-15.576
cuencas con NSE < 0,20.000,20.000


### 8.3 Otro tipo de modelo: LightGBM

**Por qué.** El LSTM necesita la ventana de 365 días completa, así que pierde el primer año después de cada hueco del registro: el 20 % de los días con caudal. LightGBM (árboles de decisión con *gradient boosting*) no usa ventanas. Describe cada día con promedios de las forzantes de los días anteriores, que se pueden calcular aunque falten algunos datos. Con pocos datos de entrenamiento, este tipo de modelo compite con el LSTM (Gauch et al., 2021). También sirve para saber si el techo de desempeño está en el modelo o en los datos.

**Diseño**, comparable con el LSTM de la sección 8:
- **Entradas (83, ninguna con caudal):** las 4 forzantes del día; la lluvia de los 3 días anteriores; promedios de lluvia de 2 a 365 días; promedios de evapotranspiración, de temperatura y del balance lluvia menos evapotranspiración; índices de lluvia antecedente (promedios exponenciales con vida media de 5 a 180 días); el día del año; y los 48 atributos. Cada promedio se calcula si hay datos en al menos la mitad de su ventana; si no, queda vacío y LightGBM lo maneja.
- **Pérdida:** error cuadrático con peso 1/(σ + 0,1)² por cuenca, el mismo del NSE* del LSTM (2.3).
- **Períodos:** entrena con 1981–2010, con un día de cada dos (días seguidos aportan casi lo mismo, y así cabe en memoria). Se detiene cuando el error en 2011–2015 (las 195 cuencas de validación) no baja en 100 árboles seguidos. Se evalúa en 2016–2022, en las 183 cuencas de prueba.
- **Comparación:** con el LSTM de la sección 8 en los mismos días, con el promedio de los dos modelos, y en todos los días con caudal, que LightGBM sí puede simular.

Entrena en unos 4 minutos en la CPU, sin GPU. Requiere `pip install lightgbm` (sección 0). Para entrenar, cambia `EJECUTAR_LIGHTGBM` a `True`.

**Resultado** (16/11/2026, `runs/camels_col_lightgbm_temporal_3e7a50`, 783 árboles):
- **LightGBM solo empata con el LSTM** en los mismos días: NSE mediano 0,390 frente a 0,394 y KGE 0,525 frente a 0,513, entrenando en minutos en la CPU en vez de horas en la GPU. Que dos modelos tan distintos lleguen al mismo techo indica que el límite está en los datos de entrada, no en el modelo.
- **El promedio de los dos modelos** sube el NSE mediano a 0,410 y mejora 127 de las 183 cuencas (prueba de Wilcoxon, p ≈ 2 × 10⁻⁷). Es la primera mejora significativa del flujo temporal. Los cambios al LSTM solo (lote 64, años 2011–2015) no lo lograron: sus errores se parecen demasiado entre sí, y los de LightGBM son distintos.
- **En todos los días con caudal**, que el LSTM no puede simular, LightGBM tiene NSE mediano 0,382 y menos cuencas con NSE negativo (17).
- **Qué usa más:** la lluvia antecedente con vida media de 20 días explica el 44 % de la ganancia, seguida de la de 60 días. El modelo se apoya sobre todo en la humedad acumulada de las últimas semanas.

In [20]:
import lightgbm as lgb
from scipy.stats import wilcoxon

EJECUTAR_LIGHTGBM = False
PARAMETROS_LGBM = {"objective": "regression", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 500,
                   "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
                   "num_threads": 4, "seed": 42, "verbose": -1}
VENTANAS_LLUVIA = [2, 3, 5, 7, 15, 30, 60, 90, 180, 365]   # días
VIDAS_MEDIAS = [5, 20, 60, 180]                            # días, índices de lluvia antecedente


def variables_lgbm(df):
    """Variables de cada día, calculadas con las forzantes de ese día y de los anteriores (nunca con caudal)."""
    p, e, tn, tx = df["pr"], df["poten_evapo"], df["t_min"], df["t_max"]
    media = lambda s, w: s.rolling(w, min_periods=max(1, w // 2)).mean()   # vacía si falta más de la mitad
    x = {"pr": p, "pet": e, "t_min": tn, "t_max": tx}
    x.update({f"pr_dia_menos_{k}": p.shift(k) for k in [1, 2, 3]})
    x.update({f"pr_media_{w}": media(p, w) for w in VENTANAS_LLUVIA})
    x.update({f"pet_media_{w}": media(e, w) for w in [7, 30, 90, 365]})
    x.update({f"balance_{w}": media(p - e, w) for w in [30, 90, 180, 365]})
    x.update({f"tmax_media_{w}": media(tx, w) for w in [7, 30]})
    x.update({f"tmin_media_{w}": media(tn, w) for w in [7, 30]})
    x.update({f"pr_antecedente_{h}": p.ewm(halflife=h).mean() for h in VIDAS_MEDIAS})
    dia = df.index.dayofyear
    x["sin_dia"], x["cos_dia"] = np.sin(2 * np.pi * dia / 365.25), np.cos(2 * np.pi * dia / 365.25)
    return pd.DataFrame(x, index=df.index)


# Variables por cuenca y días de cada conjunto (con caudal observado)
tabla = pd.read_parquet(RUTA_TABLA, columns=["gauge_id", "date", "periodo", OBJETIVO, "pr", "poten_evapo", "t_min", "t_max"])
tabla["gauge_id"] = tabla["gauge_id"].astype(str)
conjuntos, sigma = {"train": [], "validation": [], "test": []}, {}
for g, df in tabla.groupby("gauge_id", sort=True):
    df = df.set_index("date").sort_index()
    X = variables_lgbm(df).assign(**atributos_modelo.loc[g].to_dict()).astype("float32")
    y, periodo = df[OBJETIVO], df["periodo"]
    sigma[g] = y[periodo == "train"].std()   # para el peso de cada cuenca, como en el NSE*
    filas = {"train": (periodo == "train") & y.notna() & ((df.index - PERIODOS["train"][0]).days % 2 == 0),
             "validation": (periodo == "validation") & y.notna() & (g in set(validan)),
             "test": (periodo == "test") & y.notna() & (g in set(se_evaluan))}
    for k, m in filas.items():
        if m.any():
            conjuntos[k].append((g, X[m], y[m]))
del tabla
VARIABLES_LGBM = list(conjuntos["train"][0][1].columns)


def datos_lgbm(k, referencia=None):
    X = np.vstack([x.to_numpy() for _, x, _ in conjuntos[k]])
    y = np.concatenate([v.to_numpy() for _, _, v in conjuntos[k]])
    w = np.concatenate([np.full(len(v), 1 / (sigma[g] + 0.1) ** 2) for g, _, v in conjuntos[k]])
    return lgb.Dataset(X, y, weight=w, feature_name=VARIABLES_LGBM, reference=referencia, free_raw_data=True)


# Entrena una vez y guarda el modelo en runs/, con una huella de la configuración (como las corridas del LSTM)
config_lgbm = {"experiment_name": "camels_col_lightgbm_temporal", **PARAMETROS_LGBM,
               "ventanas_lluvia": VENTANAS_LLUVIA, "vidas_medias": VIDAS_MEDIAS, "variables": VARIABLES_LGBM}
archivo_lgbm = ROOT / "runs" / nombre_corrida(config_lgbm) / "modelo.txt"
if EJECUTAR_LIGHTGBM and not archivo_lgbm.exists():
    d_entrenamiento = datos_lgbm("train")
    modelo_lgbm = lgb.train(PARAMETROS_LGBM, d_entrenamiento, num_boost_round=3000,
                            valid_sets=[datos_lgbm("validation", d_entrenamiento)], valid_names=["validación"],
                            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)])
    archivo_lgbm.parent.mkdir(parents=True, exist_ok=True)
    modelo_lgbm.save_model(str(archivo_lgbm), num_iteration=modelo_lgbm.best_iteration)
    del d_entrenamiento
modelo_lgbm = lgb.Booster(model_file=str(archivo_lgbm)) if archivo_lgbm.exists() else None

if modelo_lgbm is not None:
    print(f"{archivo_lgbm.parent.name}: {modelo_lgbm.num_trees()} árboles")
    series_lgbm = {g: pd.DataFrame({"obs": y, "sim": np.clip(modelo_lgbm.predict(x.to_numpy()), 0, None)},
                                   index=x.index.rename("date"))
                   for g, x, y in conjuntos["test"]}
    comparacion_lgbm = {}
    if series_temporal:
        lstm_en = lambda g, s: series_temporal[g]["sim"].reindex(s.index)
        lgbm_mismos = {g: s[lstm_en(g, s).notna()] for g, s in series_lgbm.items() if g in series_temporal}
        promedio = {g: s.assign(sim=(s["sim"] + lstm_en(g, s)) / 2) for g, s in lgbm_mismos.items()}
        metricas_promedio = metricas(promedio)
        comparacion_lgbm = {"LSTM (8)": resumir(metricas_temporal),
                            "LightGBM, mismos días": resumir(metricas(lgbm_mismos)),
                            "Promedio LSTM y LightGBM": resumir(metricas_promedio)}
        nse = pd.DataFrame({"LSTM": metricas_temporal["NSE"], "promedio": metricas_promedio["NSE"]}).dropna()
        print(f"El promedio mejora el NSE de {(nse['promedio'] > nse['LSTM']).sum()} de {len(nse)} cuencas "
              f"(prueba de Wilcoxon: p = {wilcoxon(nse['promedio'] - nse['LSTM']).pvalue:.0e})")
    comparacion_lgbm["LightGBM, todos los días con caudal"] = resumir(metricas(series_lgbm))
    display(pd.DataFrame(comparacion_lgbm))
    importancia = pd.Series(modelo_lgbm.feature_importance("gain"), index=modelo_lgbm.feature_name())
    display((100 * importancia / importancia.sum()).nlargest(10).round(1).rename("% de la ganancia"))
else:
    print("LightGBM pendiente")
del conjuntos

camels_col_lightgbm_temporal_3e7a50: 783 árboles


El promedio mejora el NSE de 127 de 183 cuencas (prueba de Wilcoxon: p = 2e-07)


,LSTM (8),"LightGBM, mismos días",Promedio LSTM y LightGBM,"LightGBM, todos los días con caudal"
cuencas,183.000,183.000,183.000,183.000
NSE,0.394,0.390,0.410,0.382
KGE,0.513,0.525,0.518,0.515
RMSE,1.688,1.693,1.667,1.735
MAE,1.063,1.060,1.055,1.073
FHV,-29.752,-29.511,-29.777,-29.409
FLV,8.830,17.897,24.115,26.494
FMS,-10.311,-16.285,-12.727,-17.953
cuencas con NSE < 0,20.000,20.000,20.000,17.000


pr_antecedente_20    43.8
pr_antecedente_60     8.6
eroded_misce_perc     6.3
pr_antecedente_5      5.6
p_mean                3.8
aridity               3.0
balance_180           1.4
metamor_rock_perc     1.2
log_area              1.2
equi_slope            1.2
Name: % de la ganancia, dtype: float64

### 8.4 Selección de variables con LightGBM (validación 2011–2015)

**Por qué.** Probar una variable nueva en el LSTM cuesta horas de GPU; en LightGBM, unos 2 minutos de CPU. Por eso las candidatas se filtran primero con LightGBM: se agrega o se quita una variable o un grupo, se reentrena y se mide el cambio en validación. **La prueba (2016–2022) no se usa para escoger.**

**Diseño:**
- **Igual que 8.3:** las mismas variables derivadas, los mismos parámetros y los pesos del NSE*. Entrena con 1981–2010 en días alternos (1 133 707 filas) y se evalúa en las 195 cuencas de validación (309 731 días).
- **Dos cambios:**
  - 800 árboles fijos, sin parada temprana, porque en 8.3 la parada miraba la validación y aquí la validación es la que juzga;
  - 6 hilos.

  También se mide con 400 árboles, para ver si el orden de los experimentos depende del número de árboles.
- **Candidatas:**
  - **las 15 variables de ERA5-Land** (preprocesamiento, sección 2), en 7 grupos con sus medias e índices antecedentes:
    - lluvia;
    - humedad del suelo, en 4 capas y como agua total en 289 cm;
    - radiación;
    - humedad del aire: rocío, presión de vapor y déficit de presión de vapor, calculado con las temperaturas de CAMELS-COL;
    - viento y presión;
    - evaporación real;
    - escorrentía del modelo de suelo, que no es caudal observado;
  - **variables sin costo:** rango térmico, latitud y longitud de la estación, El Niño (ONI de NOAA, con la temporada que termina en el mes del día, sin mirar el futuro) y estacionalidad de la lluvia de 1981–2010;
  - **ablaciones:** quitar la evapotranspiración, las temperaturas, `eroded_misce_perc` o cada grupo de atributos.
- **Cómo se compara:** cada experimento, cuenca por cuenca, contra el promedio de 3 semillas de la línea base. Se reportan la diferencia mediana por cuenca, cuántas cuencas mejoran y la prueba de Wilcoxon.
  - **Ruido de una semilla:** cada semilla de la línea base frente al promedio de las otras dos da de −0,0001 a +0,0008 de diferencia mediana, con 95 a 110 de 195 cuencas que mejoran.
  - **Umbral:** un efecto cuenta si la diferencia supera unas 0,002, mejoran más de unas 115 cuencas y p < 0,01.
  - Lo que pasa el umbral con una semilla (42) se repite con las semillas 43 y 44.
- **Bloques en disco:** las variables se calculan una vez y se guardan por bloques en `runs/seleccion_lgbm_1811/bloques/` (unos 700 MB).

**Cómo repetirlo:**
- Cambia `EJECUTAR_SELECCION` a `True`. La celda arma los bloques (unos 10 minutos) y entrena los experimentos pendientes: 49 en total, de unos 2 minutos cada uno.
- Requiere la tabla de ERA5-Land del preprocesamiento (sección 2).
- Con la bandera en `False`, la celda muestra los resultados guardados.

**Resultado** (18/11/2026; detalle en `docs/bitacora-modelado.md`, sección 19). Promedio por cuenca de 3 semillas, frente a la línea base (NSE mediano 0,446):

| Experimento | Variables | NSE mediano | Dif. mediana por cuenca | Cuencas que mejoran (de 195) |
|---|---|---|---|---|
| **+ todo ERA5-Land, − `eroded_misce_perc`** | **116** | **0,480** | **+0,020** | **158** |
| + todo ERA5-Land | 117 | 0,480 | +0,020 | 155 |
| + suelo, radiación y lluvia de ERA5-Land | 99 | 0,479 | +0,016 | 153 |
| + lluvia de ERA5-Land | 90 | 0,457 | +0,009 | 150 |
| + radiación de ERA5-Land | 87 | 0,469 | +0,008 | 148 |
| + escorrentía de ERA5-Land | 91 | 0,455 | +0,008 | 150 |
| + rango térmico | 86 | 0,451 | +0,002 | 120 |
| + El Niño (ONI) | 84 | 0,458 | +0,001 | 106 |

- **ERA5-Land es la primera fuente nueva que mueve el resultado.** Mejora en todas las regiones, sobre todo en el Pacífico (+0,052). Sus grupos se suman: cada uno aporta algo distinto.
- **Quitar variables casi nunca mejora.** Cada grupo de atributos es redundante con los demás. La evapotranspiración es lo único cuya ausencia empeora (−0,004). `eroded_misce_perc` se puede quitar sin perder nada.
- **Recomendación:**
  - LightGBM con la base más todo ERA5-Land, sin `eroded_misce_perc` (116 variables);
  - en el LSTM, las 15 variables diarias como forzantes (8.6);
  - El Niño, el rango térmico, la ubicación y la estacionalidad no entran.

In [ ]:
import json
import time
import urllib.request

import lightgbm as lgb
from scipy.stats import wilcoxon

EJECUTAR_SELECCION = False
DIR_ERA5L = ROOT / "data" / "externos" / "era5land"            # preprocesamiento, sección 2
TABLA_ERA5L = DIR_ERA5L / "era5land_cuencas_diario.parquet"
ARCHIVO_ONI = ROOT / "data" / "externos" / "oni" / "oni.ascii.txt"
URL_ONI = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
DIR_SELECCION = ROOT / "runs" / "seleccion_lgbm_1811"
DIR_BLOQUES = DIR_SELECCION / "bloques"
ARBOLES_SELECCION, CORTE_SELECCION = 800, 400
PARAMETROS_SELECCION = {**PARAMETROS_LGBM, "num_threads": 6}   # los de 8.3; la semilla va en cada experimento
CAPAS_SUELO_MM = {1: 70, 2: 210, 3: 720, 4: 1890}              # espesor de las 4 capas de suelo de ERA5-Land
R_WGS84, EXCENTRICIDAD_WGS84 = 6378137.0, 0.0818191908426


def media_movil(s, w):
    """Media de los últimos w días; vacía si falta más de la mitad de la ventana (como en 8.3)."""
    return s.rolling(w, min_periods=max(1, w // 2)).mean()


def presion_saturacion(t):
    return 0.6108 * np.exp(17.27 * t / (t + 237.3))   # kPa, FAO-56, ec. 11


def a_grados(xy):
    """WGS 84 World Mercator (EPSG:3395, elipsoidal) a longitud y latitud en grados. Las coordenadas de las
    estaciones en atributos.csv y los polígonos de CAMELS-COL vienen en metros de esta proyección."""
    lon = np.degrees(xy[:, 0] / R_WGS84)
    t = np.exp(-xy[:, 1] / R_WGS84)
    phi = np.pi / 2 - 2 * np.arctan(t)
    for _ in range(10):   # converge a milímetros en 4 o 5 iteraciones
        e_sin = EXCENTRICIDAD_WGS84 * np.sin(phi)
        phi = np.pi / 2 - 2 * np.arctan(t * ((1 - e_sin) / (1 + e_sin)) ** (EXCENTRICIDAD_WGS84 / 2))
    return np.column_stack([lon, np.degrees(phi)])


def bloques_dinamicos(df, oni):
    """Bloques de variables diarias de una cuenca, con las forzantes de ese día y de los anteriores (nunca caudal)."""
    p, e, tn, tx = df["pr"], df["poten_evapo"], df["t_min"], df["t_max"]
    b = {}
    x = {"pr": p}
    x.update({f"pr_dia_menos_{k}": p.shift(k) for k in [1, 2, 3]})
    x.update({f"pr_media_{w}": media_movil(p, w) for w in VENTANAS_LLUVIA})
    x.update({f"pr_antecedente_{h}": p.ewm(halflife=h).mean() for h in VIDAS_MEDIAS})
    b["forz_lluvia"] = x
    x = {"pet": e}
    x.update({f"pet_media_{w}": media_movil(e, w) for w in [7, 30, 90, 365]})
    x.update({f"balance_{w}": media_movil(p - e, w) for w in [30, 90, 180, 365]})
    b["forz_etp"] = x
    x = {"t_min": tn, "t_max": tx}
    x.update({f"tmax_media_{w}": media_movil(tx, w) for w in [7, 30]})
    x.update({f"tmin_media_{w}": media_movil(tn, w) for w in [7, 30]})
    b["forz_temp"] = x
    dia = df.index.dayofyear
    b["dia"] = {"sin_dia": pd.Series(np.sin(2 * np.pi * dia / 365.25), df.index),
                "cos_dia": pd.Series(np.cos(2 * np.pi * dia / 365.25), df.index)}
    r = tx - tn
    b["rango_t"] = {"rango_t": r, "rango_t_media_7": media_movil(r, 7), "rango_t_media_30": media_movil(r, 30)}
    b["oni"] = {"oni": pd.Series(oni.reindex(df.index.to_period("M") - 1).to_numpy(), df.index)}
    # ERA5-Land (preprocesamiento, sección 2)
    ep = df["era5l_pr_mm"]
    x = {"e_pr": ep}
    x.update({f"e_pr_media_{w}": media_movil(ep, w) for w in [3, 7, 30]})
    x.update({f"e_pr_antecedente_{h}": ep.ewm(halflife=h).mean() for h in [5, 20, 60]})
    b["e_lluvia"] = x
    x = {f"e_suelo{n}": df[f"era5l_suelo{n}"] for n in range(1, 5)}
    x["e_agua_suelo_mm"] = sum(df[f"era5l_suelo{n}"] * mm for n, mm in CAPAS_SUELO_MM.items())
    b["e_suelo"] = x
    b["e_rad"] = {"e_srad": df["era5l_srad_wm2"], "e_lrad": df["era5l_lrad_wm2"],
                  "e_srad_media_7": media_movil(df["era5l_srad_wm2"], 7),
                  "e_srad_media_30": media_movil(df["era5l_srad_wm2"], 30)}
    vpd = ((presion_saturacion(tx) + presion_saturacion(tn)) / 2 - df["era5l_pvapor_kpa"]).clip(lower=0)
    b["e_humedad"] = {"e_rocio": df["era5l_rocio_c"], "e_pvapor": df["era5l_pvapor_kpa"], "e_vpd": vpd,
                      "e_vpd_media_7": media_movil(vpd, 7)}
    b["e_vientopres"] = {"e_viento": df["era5l_viento_ms"], "e_presion": df["era5l_presion_kpa"]}
    ev = df["era5l_evap_mm"]
    b["e_evap"] = {"e_evap": ev, **{f"e_evap_media_{w}": media_movil(ev, w) for w in [7, 30, 90]}}
    es = df["era5l_esc_total_mm"]
    x = {"e_esc_total": es, "e_esc_sup": df["era5l_esc_sup_mm"], "e_esc_sub": df["era5l_esc_sub_mm"]}
    x.update({f"e_esc_media_{w}": media_movil(es, w) for w in [3, 7, 30]})
    x.update({f"e_esc_antecedente_{h}": es.ewm(halflife=h).mean() for h in [20, 60]})
    b["e_escorrentia"] = x
    return b


def leer_oni():
    """ONI de NOAA (anomalía de temperatura del mar en Niño 3.4, por temporada de 3 meses). A cada día se le asigna
    la temporada centrada en el mes anterior, que termina en su mes: así no mira el futuro."""
    if not ARCHIVO_ONI.exists():
        ARCHIVO_ONI.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(URL_ONI, ARCHIVO_ONI)
    t = pd.read_csv(ARCHIVO_ONI, sep=r"\s+")
    temporadas = ["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA", "JAS", "ASO", "SON", "OND", "NDJ"]
    centro = {s: i + 1 for i, s in enumerate(temporadas)}
    return pd.Series(t["ANOM"].to_numpy(),
                     index=pd.PeriodIndex.from_fields(year=t["YR"], month=t["SEAS"].map(centro), freq="M"))


def bloques_estaticos():
    """Los 6 grupos de atributos del registro de variables, más latitud y longitud de la estación."""
    reg = pd.read_csv(ROOT / "docs" / "registro-variables.csv", encoding="utf-8-sig")
    reg = reg[reg["tipo"] == "atributo estático"].set_index("variable")["grupo"]
    nombres = {"fisiográficos": "est_fisiograficos", "climáticos": "est_climaticos", "cobertura": "est_cobertura",
               "geología": "est_geologia", "suelos": "est_suelos", "capacidad de uso": "est_capacidad"}
    assert set(reg.index) == set(atributos_modelo.columns), set(reg.index) ^ set(atributos_modelo.columns)
    bloques = {nombres[g]: atributos_modelo[list(reg[reg == g].index)] for g in nombres}
    a = pd.read_csv(DIR_BASE / "atributos.csv", dtype={"gauge_id": str}).set_index("gauge_id")
    ll = a_grados(a[["gauge_lon", "gauge_lat"]].to_numpy(float))
    bloques["latlon"] = pd.DataFrame({"lon_estacion": ll[:, 0], "lat_estacion": ll[:, 1]}, index=a.index)
    return bloques


def estacionalidad_lluvia(base):
    """Con pr de 1981–2010: índice de Walsh y Lawler, concentración y momento medio del año (seno y coseno)."""
    t = base[base["periodo"] == "train"]
    m = t.groupby(["gauge_id", t["date"].dt.month], observed=True)["pr"].mean().unstack() * 30.44   # mm por mes
    anual = m.sum(axis=1)
    si = m.sub(anual / 12, axis=0).abs().sum(axis=1) / anual
    ang = 2 * np.pi * (np.arange(1, 13) - 0.5) / 12
    cx, sy = (m * np.cos(ang)).sum(axis=1) / anual, (m * np.sin(ang)).sum(axis=1) / anual
    conc = np.hypot(cx, sy)
    return pd.DataFrame({"pr_estacionalidad": si, "pr_concentracion": conc, "pr_momento_cos": cx / conc,
                         "pr_momento_sin": sy / conc})


def tabla_con_era5l():
    """Tabla diaria de CAMELS-COL (forzantes y caudal) con las 15 variables de ERA5-Land, unidas fila por fila."""
    base = pd.read_parquet(RUTA_TABLA, columns=["gauge_id", "date", "periodo", OBJETIVO] + FORZANTES)
    base["gauge_id"] = base["gauge_id"].astype(str).astype("category")
    era = pd.read_parquet(TABLA_ERA5L)
    assert (era["gauge_id"].cat.codes.to_numpy() == base["gauge_id"].cat.codes.to_numpy()).all()
    assert list(era["gauge_id"].cat.categories.astype(str)) == list(base["gauge_id"].cat.categories)
    assert (era["date"].to_numpy() == base["date"].to_numpy()).all()
    for v in [x for x in era.columns if x.startswith("era5l_")]:
        base[v] = era[v].to_numpy()
    return base


def construir_bloques():
    """Calcula todas las variables una vez, para los días de entrenamiento y validación, y las guarda por bloques."""
    t0 = time.time()
    DIR_BLOQUES.mkdir(parents=True, exist_ok=True)
    base = tabla_con_era5l()
    oni, est = leer_oni(), bloques_estaticos()
    est["estacionalidad"] = estacionalidad_lluvia(base)
    # 1) filas de cada conjunto, como en 8.3
    y, per, g_ = base[OBJETIVO], base["periodo"], base["gauge_id"]
    sigma = y[per == "train"].groupby(g_[per == "train"], observed=True).std()
    tr = (per == "train") & y.notna() & ((base["date"] - PERIODOS["train"][0]).dt.days % 2 == 0)
    va = (per == "validation") & y.notna() & g_.astype(str).isin(set(validan))
    sel = (tr | va).to_numpy()
    filas = pd.DataFrame({"gauge_id": g_[sel].astype(str).to_numpy(), "date": base.loc[sel, "date"].to_numpy(),
                          "conjunto": np.where(tr[sel], "train", "validation"), "y": y[sel].to_numpy(),
                          "w": (1 / (g_[sel].map(sigma).astype(float) + 0.1) ** 2).to_numpy()})
    posiciones = filas.groupby("gauge_id").indices
    # 2) bloques, escritos cuenca por cuenca en arreglos en disco
    columnas, mm = None, {}
    for g, df in base.groupby("gauge_id", observed=True, sort=True):
        df = df.set_index("date")
        din = bloques_dinamicos(df, oni)
        if columnas is None:
            columnas = {b: list(x) for b, x in din.items()} | {b: list(x.columns) for b, x in est.items()}
            mm = {b: np.lib.format.open_memmap(DIR_BLOQUES / f"{b}.npy", mode="w+", dtype="float32",
                                               shape=(len(filas), len(cs))) for b, cs in columnas.items()}
        pos = posiciones.get(str(g))
        if pos is None:
            continue
        fechas_g = filas["date"].to_numpy()[pos]
        for b, x in din.items():
            mm[b][pos] = pd.DataFrame(x).reindex(fechas_g).to_numpy("float32")
        for b, x in est.items():
            mm[b][pos] = np.broadcast_to(x.loc[str(g)].to_numpy("float32"), (len(pos), x.shape[1]))
    for a in mm.values():
        a.flush()
    filas.to_parquet(DIR_SELECCION / "filas.parquet", index=False)
    (DIR_SELECCION / "columnas.json").write_text(json.dumps(columnas, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Bloques: {len(columnas)}, {sum(len(v) for v in columnas.values())} columnas, "
          f"{(filas.conjunto == 'train').sum():,} filas de entrenamiento y {(filas.conjunto == 'validation').sum():,} "
          f"de validación, {time.time() - t0:.0f} s")


if (DIR_SELECCION / "columnas.json").exists():
    columnas_bloques = json.loads((DIR_SELECCION / "columnas.json").read_text(encoding="utf-8"))
    print(f"Bloques en {DIR_BLOQUES.relative_to(ROOT)}: {len(columnas_bloques)} bloques, "
          f"{sum(len(v) for v in columnas_bloques.values())} columnas")
    display(pd.Series({b: len(v) for b, v in columnas_bloques.items()}, name="columnas").to_frame().T)
else:
    print("Bloques pendientes: cambia EJECUTAR_SELECCION a True en la celda siguiente")

Bloques en runs\seleccion_lgbm_1811\bloques: 21 bloques, 127 columnas


,forz_lluvia,forz_etp,forz_temp,dia,rango_t,oni,e_lluvia,e_suelo,e_rad,e_humedad,e_vientopres,e_evap,e_escorrentia,est_fisiograficos,est_climaticos,est_cobertura,est_geologia,est_suelos,est_capacidad,latlon,estacionalidad
columnas,18,9,6,2,3,1,7,5,4,4,2,4,8,10,7,5,6,13,7,2,4


In [ ]:
BLOQUES_BASE = ["forz_lluvia", "forz_etp", "forz_temp", "dia", "est_fisiograficos", "est_climaticos",
                "est_cobertura", "est_geologia", "est_suelos", "est_capacidad"]                 # 83 variables, como 8.3
BLOQUES_ERA5L = ["e_lluvia", "e_suelo", "e_rad", "e_humedad", "e_vientopres", "e_evap", "e_escorrentia"]
EXPERIMENTOS = {f"base_s{s}": {"bloques": BLOQUES_BASE, "semilla": s} for s in (42, 43, 44)}
EXPERIMENTOS.update({
    # Primera ronda, con una semilla (42)
    "con_era5land_todo": {"bloques": BLOQUES_BASE + BLOQUES_ERA5L},
    **{f"con_{b}": {"bloques": BLOQUES_BASE + [b]} for b in BLOQUES_ERA5L},
    **{f"con_{b}": {"bloques": BLOQUES_BASE + [b]} for b in ["rango_t", "latlon", "oni", "estacionalidad"]},
    "sin_etp": {"bloques": [b for b in BLOQUES_BASE if b != "forz_etp"]},
    "sin_temperaturas": {"bloques": [b for b in BLOQUES_BASE if b != "forz_temp"]},
    "sin_eroded_misce": {"bloques": BLOQUES_BASE, "quitar": ["eroded_misce_perc"]},
    **{f"sin_{b[4:]}": {"bloques": [x for x in BLOQUES_BASE if x != b]} for b in BLOQUES_BASE if b.startswith("est_")},
    # Segunda ronda: combinaciones de lo que pasó el umbral
    "propuesta": {"bloques": BLOQUES_BASE + BLOQUES_ERA5L + ["rango_t", "oni"], "quitar": ["eroded_misce_perc"]},
    "con_era5land_todo_sin_eroded": {"bloques": BLOQUES_BASE + BLOQUES_ERA5L, "quitar": ["eroded_misce_perc"]},
    "con_suelo_rad_lluvia": {"bloques": BLOQUES_BASE + ["e_suelo", "e_rad", "e_lluvia"]},
})
CON_TRES_SEMILLAS = ["propuesta", "con_era5land_todo", "con_era5land_todo_sin_eroded", "con_suelo_rad_lluvia",
                     "con_e_rad", "con_e_lluvia", "con_e_escorrentia", "con_e_suelo", "con_e_evap", "con_rango_t", "con_oni"]
ORDEN_SELECCION = list(EXPERIMENTOS) + [f"{e}_s{s}" for e in CON_TRES_SEMILLAS for s in (43, 44)]


def experimento_seleccion(nombre):
    """Nombre de la tabla, o nombre_sNN para repetirlo con otra semilla."""
    if nombre in EXPERIMENTOS:
        return EXPERIMENTOS[nombre]
    raiz, _, s = nombre.rpartition("_s")
    return {**EXPERIMENTOS[raiz], "semilla": int(s)}


def raiz_seleccion(nombre):
    if nombre.startswith("base_s"):
        return "base"
    r, _, s = nombre.rpartition("_s")
    return r if s.isdigit() and r in EXPERIMENTOS else nombre


def nse_kge_dias(obs, sim):
    nse = 1 - ((sim - obs) ** 2).sum() / ((obs - obs.mean()) ** 2).sum()
    r = np.corrcoef(obs, sim)[0, 1]
    kge = 1 - np.sqrt((r - 1) ** 2 + (sim.std() / obs.std() - 1) ** 2 + (sim.mean() / obs.mean() - 1) ** 2)
    return nse, kge


def matriz_seleccion(bloques, quitar, columnas, idx):
    """Filas `idx` de los bloques en disco, en el orden de los bloques y sin las columnas de `quitar`."""
    partes = []
    for b in bloques:
        js = [j for j, col in enumerate(columnas[b]) if col not in quitar]
        if js:
            partes.append(np.load(DIR_BLOQUES / f"{b}.npy", mmap_mode="r")[idx][:, js])
    return np.hstack(partes).astype("float32")


def correr_seleccion(nombres):
    """Entrena los experimentos que falten, uno a la vez, y agrega sus resultados a los CSV de DIR_SELECCION."""
    filas = pd.read_parquet(DIR_SELECCION / "filas.parquet")
    columnas = json.loads((DIR_SELECCION / "columnas.json").read_text(encoding="utf-8"))
    hechos = set(pd.read_csv(DIR_SELECCION / "resultados.csv")["experimento"]) \
        if (DIR_SELECCION / "resultados.csv").exists() else set()
    tr = np.where(filas["conjunto"].to_numpy() == "train")[0]
    va = np.where(filas["conjunto"].to_numpy() == "validation")[0]
    fv = filas.iloc[va].reset_index(drop=True)
    for nombre in [n for n in nombres if n not in hechos]:
        exp, t0 = experimento_seleccion(nombre), time.time()
        quitar = exp.get("quitar", [])
        nombres_col = [col for b in exp["bloques"] for col in columnas[b] if col not in quitar]
        d = lgb.Dataset(matriz_seleccion(exp["bloques"], quitar, columnas, tr), filas["y"].to_numpy()[tr],
                        weight=filas["w"].to_numpy()[tr], feature_name=nombres_col, free_raw_data=True)
        d.construct()
        modelo = lgb.train({**PARAMETROS_SELECCION, "seed": exp.get("semilla", 42)}, d,
                           num_boost_round=ARBOLES_SELECCION)
        del d
        Xv = matriz_seleccion(exp["bloques"], quitar, columnas, va)
        pred = {a: np.clip(modelo.predict(Xv, num_iteration=a), 0, None) for a in (CORTE_SELECCION, ARBOLES_SELECCION)}
        por_cuenca = []
        for g, idx in fv.groupby("gauge_id").indices.items():
            fila = {"experimento": nombre, "gauge_id": g, "dias": len(idx)}
            for a, p in pred.items():
                fila[f"nse_{a}"], fila[f"kge_{a}"] = nse_kge_dias(fv["y"].to_numpy()[idx], p[idx])
            por_cuenca.append(fila)
        por_cuenca = pd.DataFrame(por_cuenca)
        imp = pd.Series(modelo.feature_importance("gain"), index=nombres_col)
        imp = (100 * imp / imp.sum()).round(3)
        fila = {"experimento": nombre, "bloques": "+".join(exp["bloques"]), "quitar": ",".join(quitar),
                "semilla": exp.get("semilla", 42), "variables": len(nombres_col), "cuencas": len(por_cuenca),
                "nse_mediano": por_cuenca[f"nse_{ARBOLES_SELECCION}"].median(),
                "kge_mediano": por_cuenca[f"kge_{ARBOLES_SELECCION}"].median(),
                f"nse_mediano_{CORTE_SELECCION}": por_cuenca[f"nse_{CORTE_SELECCION}"].median(),
                "minutos": (time.time() - t0) / 60,
                "top5_ganancia": "; ".join(f"{k} {v:.1f}" for k, v in imp.nlargest(5).items())}
        for archivo, df in [("resultados.csv", pd.DataFrame([fila])), ("resultados_cuencas.csv", por_cuenca),
                            ("importancia.csv", imp.rename("pct_ganancia").rename_axis("variable").reset_index()
                             .assign(experimento=nombre))]:
            df.to_csv(DIR_SELECCION / archivo, mode="a", header=not (DIR_SELECCION / archivo).exists(), index=False)
        print(f"{nombre}: {len(nombres_col)} variables, NSE mediano {fila['nse_mediano']:.4f}, {fila['minutos']:.1f} min")


def resumen_seleccion(guardar=False):
    """Cada experimento (promedio por cuenca de sus semillas) frente al promedio por cuenca de las 3 semillas base."""
    r = pd.read_csv(DIR_SELECCION / "resultados.csv").drop_duplicates("experimento", keep="last").set_index("experimento")
    pc = pd.read_csv(DIR_SELECCION / "resultados_cuencas.csv", dtype={"gauge_id": str})
    pc = pc.drop_duplicates(["experimento", "gauge_id"], keep="last")
    nse = {a: pc.pivot(index="gauge_id", columns="experimento", values=f"nse_{a}")
           for a in (CORTE_SELECCION, ARBOLES_SELECCION)}
    kge = pc.pivot(index="gauge_id", columns="experimento", values=f"kge_{ARBOLES_SELECCION}")
    semillas = [c for c in nse[ARBOLES_SELECCION].columns if c.startswith("base_s")]
    ref = {a: t[semillas].mean(axis=1) for a, t in nse.items()}
    n800 = nse[ARBOLES_SELECCION]
    ruido = [n800[s].median() - n800[[o for o in semillas if o != s]].mean(axis=1).median() for s in semillas]
    grupos = {}
    for e in n800.columns:
        grupos.setdefault(raiz_seleccion(e), []).append(e)
    salida = []
    for g, es in grupos.items():
        if g == "base":
            continue
        x = n800[es].mean(axis=1)
        d = (x - ref[ARBOLES_SELECCION]).dropna()
        d400 = (nse[CORTE_SELECCION][es].mean(axis=1) - ref[CORTE_SELECCION]).dropna()
        salida.append({"experimento": g, "semillas": len(es), "variables": r.loc[es[0], "variables"],
                       "nse_mediano": x.median(), "kge_mediano": kge[es].mean(axis=1).median(),
                       "dif_mediana_por_cuenca": d.median(), "cuencas_mejoran": int((d > 0).sum()), "cuencas": len(d),
                       "p_wilcoxon": wilcoxon(d).pvalue if (d != 0).any() else np.nan,
                       f"dif_mediana_por_cuenca_{CORTE_SELECCION}": d400.median()})
    t = pd.DataFrame(salida).set_index("experimento")
    t = t.loc[[e for e in EXPERIMENTOS if e in t.index] + [e for e in t.index if e not in EXPERIMENTOS]]
    if guardar:
        t.to_csv(DIR_SELECCION / "resumen.csv")
    print(f"Línea base: NSE mediano del promedio por cuenca de las 3 semillas {ref[ARBOLES_SELECCION].median():.4f}; "
          f"por semilla {[round(n800[s].median(), 4) for s in semillas]}; ruido de una semilla frente a las otras dos "
          f"(diferencia de medianas) {[round(x, 4) for x in ruido]}")
    return t


if EJECUTAR_SELECCION:
    if not (DIR_SELECCION / "columnas.json").exists():
        construir_bloques()
    correr_seleccion(ORDEN_SELECCION)

if (DIR_SELECCION / "resultados_cuencas.csv").exists():
    tabla_seleccion = resumen_seleccion(guardar=EJECUTAR_SELECCION)
    display(tabla_seleccion[["semillas", "variables", "nse_mediano", "kge_mediano", "dif_mediana_por_cuenca",
                             "cuencas_mejoran", "cuencas", "p_wilcoxon", f"dif_mediana_por_cuenca_{CORTE_SELECCION}"]]
            .round(4))
else:
    print("Selección pendiente")

Línea base: NSE mediano del promedio por cuenca de las 3 semillas 0.4459; por semilla [np.float64(0.451), np.float64(0.4477), np.float64(0.4516)]; ruido de una semilla frente a las otras dos (diferencia de medianas) [np.float64(0.0077), np.float64(-0.0029), np.float64(0.0003)]

,semillas,variables,nse_mediano,kge_mediano,dif_mediana_por_cuenca,cuencas_mejoran,cuencas,p_wilcoxon,dif_mediana_por_cuenca_400
experimento,,,,,,,,,
con_era5land_todo,3,117,0.4797,0.5900,0.0198,155,195,0.0000,0.0162
con_e_lluvia,3,90,0.4567,0.5717,0.0091,150,195,0.0000,0.0083
con_e_suelo,3,88,0.4647,0.5725,0.0044,137,195,0.0000,0.0041
con_e_rad,3,87,0.4686,0.5812,0.0083,148,195,0.0000,0.0098
con_e_humedad,1,87,0.4538,0.5635,0.0016,113,195,0.0069,0.0011
con_e_vientopres,1,85,0.4525,0.5723,0.0029,122,195,0.0000,0.0031
con_e_evap,3,87,0.4595,0.5698,0.0045,133,195,0.0000,0.0043
con_e_escorrentia,3,91,0.4550,0.5716,0.0082,150,195,0.0000,0.0073
con_rango_t,3,86,0.4506,0.5664,0.0021,120,195,0.0007,0.0020


### 8.5 Importancia por permutación en el LSTM

**Qué.** Se toma el LSTM ya entrenado (sección 8, época 3), se revuelve una entrada o un grupo y se mide cuánto cae el NSE en validación 2011–2015 (195 cuencas). No se reentrena: mide de qué depende el modelo que ya tenemos.
- **Forzantes:** se permutan los días de 2010–2015 dentro de cada cuenca (la ventana de 365 días más la validación), solo entre días con dato, así que los huecos quedan donde estaban.
- **Atributos:** se permutan entre cuencas, con la misma permutación para todas las columnas del grupo.
- **Evaluador:** el de NeuralHydrology, sin cambios. Solo se intercepta la lectura de datos, y al terminar se restaura.

Son 12 evaluaciones, unos 16 minutos de GPU. Para repetir, cambia `EJECUTAR_PERMUTACION` a `True`.

**Resultado** (18/11/2026; bitácora, sección 20). NSE mediano sin revolver: 0,460.

| Se revuelve | NSE mediano | Caída mediana por cuenca | Cuencas que empeoran más de 0,005 | En LightGBM, quitarlo (8.4) |
|---|---|---|---|---|
| `pr` | −0,107 | −0,531 | 189 | (no se probó: es la entrada principal) |
| Los 48 atributos | 0,122 | −0,326 | 178 | — |
| Fisiográficos (10) | 0,332 | −0,053 | 135 | −0,001 |
| Climáticos (7) | 0,365 | −0,051 | 143 | 0,000 |
| Suelos (13) | 0,382 | −0,038 | 125 | 0,000 |
| `poten_evapo` | 0,413 | −0,024 | 135 | −0,004 |
| `t_min` y `t_max` | 0,440 | −0,007 | 103 | 0,000 |
| `eroded_misce_perc` | 0,460 | 0,000 | 14 | +0,001 |

- **El LSTM se apoya mucho más en los atributos que LightGBM.** Cada grupo resta de 0,02 a 0,05 por cuenca; en LightGBM, quitar un grupo no cambiaba nada.
  - Parte de la diferencia es del método: al revolver se crean cuencas imposibles, y el modelo no puede compensar con otras variables como cuando se reentrena.
  - Por eso los atributos no se quitan sin probarlos en cuencas no vistas.
- **El LSTM usa `poten_evapo` (−0,024) más que las temperaturas (−0,007),** aunque sale de ellas: le sirve la combinación ya hecha.
- **`eroded_misce_perc` no la usa** (caída nula). Junto con 8.4, se quita en los dos modelos.

In [ ]:
import zlib

import neuralhydrology.datasetzoo.genericdataset as gd
from neuralhydrology.evaluation import get_tester

EJECUTAR_PERMUTACION = False
ARCHIVO_PERMUTACION = DIR_SELECCION / "permutacion_lstm.csv"
VENTANA_PERMUTACION = (pd.Timestamp("2010-01-01"), PERIODOS["validation"][1])   # 365 días previos + validación


def experimentos_permutacion(cfg):
    """{nombre: (forzantes que se revuelven, atributos que se revuelven)}; los grupos salen del registro de variables."""
    reg = pd.read_csv(ROOT / "docs" / "registro-variables.csv", encoding="utf-8-sig")
    grupo = reg[reg["tipo"] == "atributo estático"].set_index("variable")["grupo"]
    estaticas = list(cfg.static_attributes)
    exps = {"base": ([], []), "pr": (["pr"], []), "poten_evapo": (["poten_evapo"], []),
            "temperaturas": (["t_min", "t_max"], [])}
    for g in grupo.unique():
        vs = [v for v in estaticas if grupo.get(v) == g]
        if vs:
            exps[f"est_{g}"] = ([], vs)
    exps["eroded_misce_perc"] = ([], ["eroded_misce_perc"])
    exps["todas_estaticas"] = ([], estaticas)
    return exps


def permutar_lstm(run_dir, epoca):
    """Evalúa la validación con cada entrada revuelta y agrega el NSE por cuenca a ARCHIVO_PERMUTACION."""
    cfg = Config(run_dir / "config.yml")
    cfg.update_config({"cache_validation_data": False, "verbose": 0})
    leer_series, leer_atributos = gd.load_timeseries, gd.load_attributes
    estado = {"dinamicas": [], "atributos": None}

    def series_revueltas(data_dir, basin):
        df = leer_series(data_dir=data_dir, basin=basin)
        for v in estado["dinamicas"]:
            m = (df.index >= VENTANA_PERMUTACION[0]) & (df.index <= VENTANA_PERMUTACION[1]) & df[v].notna().to_numpy()
            rng = np.random.default_rng(zlib.crc32(f"{basin}-{v}".encode()))
            df.loc[m, v] = rng.permutation(df.loc[m, v].to_numpy())
        return df

    def atributos_revueltos(data_dir, basins=None):
        df = estado["atributos"] if estado["atributos"] is not None else leer_atributos(data_dir)
        return df.loc[basins] if basins else df

    todos = leer_atributos(cfg.data_dir)
    hechos = set(pd.read_csv(ARCHIVO_PERMUTACION)["experimento"]) if ARCHIVO_PERMUTACION.exists() else set()
    gd.load_timeseries, gd.load_attributes = series_revueltas, atributos_revueltos
    logging.disable(logging.WARNING)
    try:
        for nombre, (din, est) in experimentos_permutacion(cfg).items():
            if nombre in hechos:
                continue
            t0 = time.time()
            estado["dinamicas"], estado["atributos"] = din, None
            if est:
                a = todos.copy()
                a[est] = todos[est].to_numpy()[np.random.default_rng(zlib.crc32(nombre.encode())).permutation(len(a))]
                estado["atributos"] = a
            res = get_tester(cfg, run_dir, period="validation", init_model=True).evaluate(
                epoch=epoca, save_results=False, metrics=["NSE"])
            t = pd.DataFrame({"experimento": nombre, "gauge_id": list(res),
                              "nse": [res[b]["1D"].get("NSE", np.nan) for b in res]})
            t.to_csv(ARCHIVO_PERMUTACION, mode="a", header=not ARCHIVO_PERMUTACION.exists(), index=False)
            print(f"{nombre}: NSE mediano {t['nse'].median():.4f}, {(time.time() - t0) / 60:.1f} min")
    finally:   # deja NeuralHydrology como estaba
        gd.load_timeseries, gd.load_attributes = leer_series, leer_atributos
        logging.disable(logging.NOTSET)


if EJECUTAR_PERMUTACION and run_temporal is not None:
    permutar_lstm(run_temporal, mejor_epoca(run_temporal))

if ARCHIVO_PERMUTACION.exists():
    nse_perm = pd.read_csv(ARCHIVO_PERMUTACION, dtype={"gauge_id": str}).pivot(index="gauge_id", columns="experimento",
                                                                               values="nse")
    caida = nse_perm.sub(nse_perm["base"], axis=0)
    tabla_permutacion = pd.DataFrame({"NSE mediano": nse_perm.median(), "caída mediana por cuenca": caida.median(),
                                      "cuencas que empeoran más de 0,005": (caida < -0.005).sum()})
    print(f"NSE mediano sin revolver: {nse_perm['base'].median():.3f} ({nse_perm['base'].notna().sum()} cuencas)")
    display(tabla_permutacion.drop("base").sort_values("caída mediana por cuenca").round(3))
else:
    print("Permutación pendiente")

NSE mediano sin revolver: 0.460 (195 cuencas)


,NSE mediano,caída mediana por cuenca,"cuencas que empeoran más de 0,005"
experimento,,,
pr,-0.107,-0.531,189
todas_estaticas,0.122,-0.326,178
est_fisiográficos,0.332,-0.053,135
est_climáticos,0.365,-0.051,143
est_suelos,0.382,-0.038,125
est_capacidad de uso,0.393,-0.029,126
est_geología,0.407,-0.028,130
poten_evapo,0.413,-0.024,135
est_cobertura,0.414,-0.023,132


### 8.6 Confirmación con el LSTM: ERA5-Land como forzantes

**Qué.** LightGBM (8.4) dice que ERA5-Land ayuda; falta saber si el LSTM también lo aprovecha. Se comparan dos configuraciones, iguales a la de la sección 8 salvo en las entradas:
- **`base` (C1):** las 4 forzantes y los 48 atributos, leyendo `data/processed`;
- **`era5l` (C2):** las 4 forzantes más las 15 diarias de ERA5-Land, y 47 atributos (sin `eroded_misce_perc`), leyendo `data/processed_era5land` (preprocesamiento, 2.5).

**Diseño:**
- semillas 42, 43 y 44, con 3 épocas fijas, sin parada temprana y con tasa de aprendizaje constante (la mejor época de la sección 8 fue la 3);
- se comparan en validación 2011–2015, en la época 3, fijada de antemano; las épocas 1 y 2 quedan como referencia;
- se reportan las diferencias por cuenca del promedio de las 3 semillas, del ensamble (el promedio de las 3 simulaciones) y semilla por semilla;
- el ruido se mide con pares de la misma configuración y distinta semilla;
- **control:** `base` con semilla 42 es la configuración de la sección 8, así que debe reproducir su época 3.

**Costo:** 6 corridas, de 49 minutos (`base`) a 59 (`era5l`): unas 5,5 horas de GPU.

**Cómo repetirlo:**
- Cambia `EJECUTAR_CONFIRMACION` a `True`. Cada corrida se entrena en un proceso aparte, que libera la RAM al terminar, y se saltan las que ya están.
- Las corridas quedan en `runs/confirmacion_{base,era5l}_s{42,43,44}_<fecha>`.

**Resultado** (19/11/2026; bitácora, sección 21). Validación, 195 cuencas, época 3:

| Comparación | NSE `base` | NSE `era5l` | Diferencia mediana por cuenca | Cuencas que mejoran | p (Wilcoxon) |
|---|---|---|---|---|---|
| **Promedio de las 3 semillas** | 0,452 | **0,491** | **+0,024** | **145** | 2e-12 |
| Ensamble de las 3 semillas | 0,468 | **0,505** | +0,024 | 146 | 9e-13 |
| Semilla 42 / 43 / 44 | 0,460 / 0,445 / 0,446 | 0,482 / 0,489 / 0,509 | +0,027 / +0,023 / +0,028 | 127 / 132 / 131 | ≤ 2e-5 |
| Ruido: misma configuración, otra semilla (6 pares) | — | — | −0,012 a 0,000 | 82 a 97 | 0,006 a 0,93 |

- **La mejora es clara:** la sostienen las 3 semillas, y es unas 2 veces el mayor cambio entre semillas y unas 10 veces el cambio típico. También aparece en las épocas 1 y 2.
- **Es pareja:** todas las regiones mejoran (de +0,023 a +0,050), y también todos los tramos de NSE.
  - En el Pacífico pesa mucho la semilla: −0,057 con la 42 y +0,087 con la 43.
- **Coincide con LightGBM** (+0,020 en 8.4).
- **Las 13 cuencas con NSE negativo suben, pero 11 siguen por debajo de 0.**
- **El control se cumple:** `base` con semilla 42 reproduce la sección 8 cuenca por cuenca. El entrenamiento es determinista.

In [ ]:
import ctypes
import subprocess
import sys

EJECUTAR_CONFIRMACION = False
DIR_BASE_ERA5L = ROOT / "data" / "processed_era5land"        # preprocesamiento, sección 2.5
VARIABLES_ERA5L = ["era5l_pr_mm", "era5l_srad_wm2", "era5l_lrad_wm2", "era5l_evap_mm", "era5l_esc_total_mm",
                   "era5l_esc_sup_mm", "era5l_esc_sub_mm", "era5l_suelo1", "era5l_suelo2", "era5l_suelo3",
                   "era5l_suelo4", "era5l_rocio_c", "era5l_pvapor_kpa", "era5l_viento_ms", "era5l_presion_kpa"]
SEMILLAS_CONFIRMACION = [42, 43, 44]
EPOCAS_CONFIRMACION = 3                                       # la mejor época de la sección 8
DIR_CONFIRMACION = ROOT / "runs" / "confirmacion_lstm_1811"
region_de = resumen.set_index("gauge_id")["region"]


def config_confirmacion(conf, semilla):
    """`base`: la configuración de la sección 8. `era5l`: más las 15 variables de ERA5-Land, sin eroded_misce_perc."""
    c = {**config_temporal, "experiment_name": f"confirmacion_{conf}_s{semilla}", "seed": semilla,
         "epochs": EPOCAS_CONFIRMACION, "early_stopping": False, "save_validation_results": True, "verbose": 0}
    for k in ("patience_early_stopping", "minimum_epochs_before_early_stopping",
              "patience_dynamic_learning_rate", "factor_dynamic_learning_rate"):
        c.pop(k)
    if conf == "era5l":
        c.update(data_dir="data/processed_era5land", dynamic_inputs=FORZANTES + VARIABLES_ERA5L,
                 static_attributes=[a for a in ATRIBUTOS if a != "eroded_misce_perc"])
    return c


def corrida_confirmacion(conf, semilla):
    """Carpeta más reciente de la corrida, si ya tiene la validación de la última época."""
    hechas = sorted(p for p in (ROOT / "runs").glob(f"confirmacion_{conf}_s{semilla}_*")
                    if (p / "validation" / f"model_epoch{EPOCAS_CONFIRMACION:03d}" / "validation_metrics.csv").exists())
    return hechas[-1] if hechas else None


def entrenar_confirmacion():
    """Entrena las corridas que falten, cada una en un proceso aparte (libera la RAM al terminar), alternando
    configuraciones para que, si se corta, queden pares completos de la misma semilla."""
    DIR_CONFIRMACION.mkdir(parents=True, exist_ok=True)
    if sys.platform == "win32":   # evita que el equipo se suspenda mientras entrena
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)
    try:
        for semilla in SEMILLAS_CONFIRMACION:
            for conf in ("era5l", "base"):
                if corrida_confirmacion(conf, semilla) is not None:
                    continue
                nombre = f"confirmacion_{conf}_s{semilla}"
                ruta = DIR_CONFIGS_CORRIDAS / f"{nombre}.yml"
                escribir_config(config_confirmacion(conf, semilla), ruta)
                t0 = time.time()
                with open(DIR_CONFIRMACION / f"{nombre}.log", "a", encoding="utf-8") as f:
                    subprocess.run([sys.executable, "-m", "neuralhydrology.nh_run", "train", "--config-file", str(ruta)],
                                   stdout=f, stderr=subprocess.STDOUT, cwd=ROOT, check=True)
                print(f"{nombre}: {(time.time() - t0) / 60:.0f} min")
    finally:
        if sys.platform == "win32":
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)


def nse_validacion(run_dir, epoca):
    m = pd.read_csv(Path(run_dir) / "validation" / f"model_epoch{epoca:03d}" / "validation_metrics.csv",
                    dtype={"basin": str})
    return m.set_index("basin")["NSE"]


def sim_validacion(run_dir, epoca):
    """{cuenca: (observado, simulado recortado en cero)} de la validación guardada."""
    with open(Path(run_dir) / "validation" / f"model_epoch{epoca:03d}" / "validation_results.p", "rb") as f:
        res = pickle.load(f)
    salida = {}
    for b, r in res.items():
        x = r["1D"]["xr"].isel(time_step=-1)
        salida[str(b)] = (x[f"{OBJETIVO}_obs"].to_numpy(), np.clip(x[f"{OBJETIVO}_sim"].to_numpy(), 0, None))
    return salida


def nse_serie(obs, sim):
    m = np.isfinite(obs) & np.isfinite(sim)
    if m.sum() < 2:
        return np.nan
    o, s = obs[m], sim[m]
    return 1 - ((s - o) ** 2).sum() / ((o - o.mean()) ** 2).sum()


def diferencia_por_cuenca(nombre, a, b):
    """b − a por cuenca: medianas, diferencia mediana, cuencas que mejoran y prueba de Wilcoxon."""
    d = (b - a).dropna()
    return {"comparación": nombre, "mediana a": a[d.index].median(), "mediana b": b[d.index].median(),
            "dif. mediana por cuenca": d.median(), "cuencas que mejoran": int((d > 0).sum()), "cuencas": len(d),
            "p (Wilcoxon)": wilcoxon(d).pvalue if (d != 0).any() else np.nan}


if EJECUTAR_CONFIRMACION:
    entrenar_confirmacion()

corridas_conf = {(c, s): corrida_confirmacion(c, s) for c in ("base", "era5l") for s in SEMILLAS_CONFIRMACION}
if all(r is not None for r in corridas_conf.values()):
    e = EPOCAS_CONFIRMACION
    nse_conf = {(c, s, ep): nse_validacion(r, ep) for (c, s), r in corridas_conf.items() for ep in range(1, e + 1)}
    print("NSE mediano de validación por época:")
    display(pd.DataFrame({f"época {ep}": {f"{c}, semilla {s}": nse_conf[(c, s, ep)].median() for c, s in corridas_conf}
                          for ep in range(1, e + 1)}).round(3))
    if run_temporal is not None:
        d1 = (nse_conf[("base", 42, e)] - nse_validacion(run_temporal, e)).abs()
        print(f"Control: base con semilla 42 frente a la sección 8, época {e}: diferencia absoluta por cuenca "
              f"mediana {d1.median():.4f}, máxima {d1.max():.4f}")
    prom = {c: pd.concat([nse_conf[(c, s, e)] for s in SEMILLAS_CONFIRMACION], axis=1).mean(axis=1)
            for c in ("base", "era5l")}
    ens = {}
    for c in ("base", "era5l"):
        sims = [sim_validacion(corridas_conf[(c, s)], e) for s in SEMILLAS_CONFIRMACION]
        ens[c] = pd.Series({b: nse_serie(sims[0][b][0], np.mean([x[b][1] for x in sims], axis=0)) for b in sims[0]})
    filas_conf = [diferencia_por_cuenca("era5l − base, promedio de las 3 semillas", prom["base"], prom["era5l"]),
                  diferencia_por_cuenca("era5l − base, ensamble de las 3 semillas", ens["base"], ens["era5l"])]
    filas_conf += [diferencia_por_cuenca(f"era5l − base, semilla {s}", nse_conf[("base", s, e)],
                                         nse_conf[("era5l", s, e)]) for s in SEMILLAS_CONFIRMACION]
    filas_conf += [diferencia_por_cuenca(f"ruido {c}: semilla {s2} − semilla {s1}", nse_conf[(c, s1, e)],
                                         nse_conf[(c, s2, e)])
                   for c in ("base", "era5l") for i, s1 in enumerate(SEMILLAS_CONFIRMACION)
                   for s2 in SEMILLAS_CONFIRMACION[i + 1:]]
    print(f"Comparaciones en la época {e} (a = base o la primera semilla, b = era5l o la segunda):")
    display(pd.DataFrame(filas_conf).set_index("comparación").round(4))
    dif_conf = prom["era5l"] - prom["base"]
    por_region = dif_conf.groupby(region_de.reindex(dif_conf.index))
    tramo = pd.cut(prom["base"], [-np.inf, 0, 0.3, 0.5, 0.7, np.inf],
                   labels=["< 0", "0 a 0,3", "0,3 a 0,5", "0,5 a 0,7", "≥ 0,7"], right=False)
    print("Diferencia por cuenca (promedio de semillas), por región y por tramo del NSE de base:")
    display(pd.DataFrame({"cuencas": por_region.size(), "dif. mediana": por_region.median().round(3),
                          "mejoran": por_region.apply(lambda x: int((x > 0).sum()))}))
    display(dif_conf.groupby(tramo, observed=True).agg(cuencas="size", dif_mediana="median",
                                                       mejoran=lambda x: int((x > 0).sum())).round(3))
else:
    print("Confirmación pendiente:", [f"{c}_s{s}" for (c, s), r in corridas_conf.items() if r is None])

NSE mediano de validación por época:


,época 1,época 2,época 3
"base, semilla 42",0.433,0.439,0.460
"base, semilla 43",0.404,0.437,0.445
"base, semilla 44",0.412,0.449,0.446
"era5l, semilla 42",0.464,0.483,0.482
"era5l, semilla 43",0.449,0.489,0.489
"era5l, semilla 44",0.463,0.486,0.509


Control: base con semilla 42 frente a la sección 8, época 3: diferencia absoluta por cuenca mediana 0.0000, máxima 0.0000


Comparaciones en la época 3 (a = base o la primera semilla, b = era5l o la segunda):


,mediana a,mediana b,dif. mediana por cuenca,cuencas que mejoran,cuencas,p (Wilcoxon)
comparación,,,,,,
"era5l − base, promedio de las 3 semillas",0.4517,0.4911,0.0245,145,195,0.0000
"era5l − base, ensamble de las 3 semillas",0.4676,0.5046,0.0243,146,195,0.0000
"era5l − base, semilla 42",0.4600,0.4823,0.0273,127,195,0.0000
"era5l − base, semilla 43",0.4450,0.4891,0.0231,132,195,0.0000
"era5l − base, semilla 44",0.4459,0.5091,0.0277,131,195,0.0000
ruido base: semilla 43 − semilla 42,0.4600,0.4450,-0.0018,94,195,0.5884
ruido base: semilla 44 − semilla 42,0.4600,0.4459,-0.0052,87,195,0.0471
ruido base: semilla 44 − semilla 43,0.4450,0.4459,-0.0116,82,195,0.0061
ruido era5l: semilla 43 − semilla 42,0.4823,0.4891,-0.0023,92,195,0.9273


Diferencia por cuenca (promedio de semillas), por región y por tramo del NSE de base:


,cuencas,dif. mediana,mejoran
region,,,
Amazon,9,0.038,9
Caribbean,25,0.023,19
Magdalena-Cauca,113,0.023,83
Orinoco,31,0.023,22
Pacific,17,0.050,12


,cuencas,dif_mediana,mejoran
< 0,13,0.063,9
"0 a 0,3",42,0.025,29
"0,3 a 0,5",60,0.023,48
"0,5 a 0,7",48,0.028,36
"≥ 0,7",32,0.019,23


### 8.7 Evaluación final en la prueba 2016–2022

**Modelo final, fijado antes de mirar la prueba** (bitácora, sección 22): el promedio de dos ensambles, en los días que el LSTM puede simular.
- **LSTM:** el ensamble de las 3 corridas `era5l` de 8.6 (época 3), es decir, el promedio de sus 3 simulaciones. No se reentrena.
- **LightGBM:** el ensamble de 3 modelos con la configuración recomendada en 8.4: base + ERA5-Land − `eroded_misce_perc` (116 variables), 800 árboles y semillas 42, 43 y 44.

**Cómo se evalúa:** igual que 8.3.
- Se usan las 183 cuencas de prueba, en los días con caudal observado, con al menos 365 días evaluables por cuenca.
- Las métricas son las de NeuralHydrology (sección 6), con la simulación recortada en cero, y se reportan sus medianas entre cuencas.
- Todos los modelos se comparan en los mismos días: los que simulan los tres LSTM (T1, C1 y C2), que resultan idénticos.

**Referencias**, que se reportan sin usarlas para escoger:
- T1, T4 y T5, recalculadas para comprobar que el cálculo reproduce 0,394, 0,390 y 0,410;
- C1 en ensamble de 3 semillas, solo y con LightGBM base (3 semillas), para medir cuánto aporta ERA5-Land en la prueba.

**Después de esta evaluación no se cambia nada del modelo mirando la prueba.**

**Cómo repetirlo:**
- Con `EJECUTAR_PRUEBA_FINAL` en `True`, la celda entrena los 6 LightGBM, de unos 2 minutos cada uno, y comprueba que reproducen su NSE de validación de 8.4. Luego predice la prueba.
- La evaluación de la prueba del LSTM la hace `evaluar` (sección 6), unos 2 minutos por corrida.
- Con la bandera en `False`, calcula las métricas con las simulaciones guardadas, en unos 3 minutos.

**Resultado** (19/11/2026). Prueba 2016–2022, 183 cuencas, mismos días:

| Modelo | NSE | KGE | NSE < 0 | NSE ≥ 0,5 |
|---|---|---|---|---|
| T5: LSTM + LightGBM sin ERA5-Land (mejor hasta el 17/11) | 0,410 | 0,518 | 20 | 59 |
| C2: LSTM con ERA5-Land, ensamble de 3 semillas | 0,432 | **0,539** | **14** | 72 |
| LightGBM con ERA5-Land, 3 semillas | 0,417 | 0,529 | 18 | 64 |
| **FINAL: C2 + LightGBM con ERA5-Land** | **0,437** | 0,536 | 17 | **72** |
| Meta de la propuesta | 0,70 | 0,70 | — | — |

- **Frente a T5:** +0,020 de NSE por cuenca; mejoran 127 de 183 cuencas (p = 1e-8). El KGE casi no cambia (+0,002, p = 0,38).
- **ERA5-Land ayuda en la prueba lo mismo que en validación:** +0,024 en el LSTM (validación: +0,024) y +0,015 en LightGBM (validación: +0,020). La selección no se sobreajustó.
- **Con ERA5-Land, LightGBM casi no agrega al LSTM** (+0,002). El LSTM solo es casi igual de bueno y más simple.
- **Por región** (T5 → final):

  | Región | T5 | Final |
  |---|---|---|
  | Amazonas | 0,64 | 0,70 |
  | Orinoco | 0,69 | 0,71 |
  | Pacífico | 0,43 | 0,50 |
  | Caribe | 0,35 | 0,38 |
  | Magdalena-Cauca | 0,36 | 0,38 |

  El techo sigue en Magdalena-Cauca y el Caribe, las regiones con deriva en la relación lluvia-caudal y balance inconsistente.
- **Sigue lejos de la meta de 0,70.**

In [ ]:
EJECUTAR_PRUEBA_FINAL = False
DIR_PRUEBA_FINAL = ROOT / "runs" / "prueba_final_1911"
LGBM_FINAL = {"base": (BLOQUES_BASE, []), "era5l": (BLOQUES_BASE + BLOQUES_ERA5L, ["eroded_misce_perc"])}
VALIDACION_8_4 = {("base", 42): "base_s42", ("base", 43): "base_s43", ("base", 44): "base_s44",
                  ("era5l", 42): "con_era5land_todo_sin_eroded", ("era5l", 43): "con_era5land_todo_sin_eroded_s43",
                  ("era5l", 44): "con_era5land_todo_sin_eroded_s44"}


def columnas_lgbm_final(conf, columnas):
    bloques, quitar = LGBM_FINAL[conf]
    return [col for b in bloques for col in columnas[b] if col not in quitar]


def entrenar_lgbm_final():
    """Entrena los LightGBM que falten (como en 8.4) y comprueba que reproducen su NSE de validación de 8.4."""
    DIR_PRUEBA_FINAL.mkdir(parents=True, exist_ok=True)
    columnas = json.loads((DIR_SELECCION / "columnas.json").read_text(encoding="utf-8"))
    filas = pd.read_parquet(DIR_SELECCION / "filas.parquet")
    tr = np.where(filas["conjunto"].to_numpy() == "train")[0]
    va = np.where(filas["conjunto"].to_numpy() == "validation")[0]
    fv = filas.iloc[va].reset_index(drop=True)
    ref = pd.read_csv(DIR_SELECCION / "resultados.csv").set_index("experimento")["nse_mediano"]
    for conf, (bloques, quitar) in LGBM_FINAL.items():
        for s in SEMILLAS_CONFIRMACION:
            archivo = DIR_PRUEBA_FINAL / f"lgbm_{conf}_s{s}.txt"
            if archivo.exists():
                continue
            d = lgb.Dataset(matriz_seleccion(bloques, quitar, columnas, tr), filas["y"].to_numpy()[tr],
                            weight=filas["w"].to_numpy()[tr], feature_name=columnas_lgbm_final(conf, columnas),
                            free_raw_data=True)
            d.construct()
            modelo = lgb.train({**PARAMETROS_SELECCION, "seed": s}, d, num_boost_round=ARBOLES_SELECCION)
            del d
            modelo.save_model(str(archivo))
            pv = np.clip(modelo.predict(matriz_seleccion(bloques, quitar, columnas, va)), 0, None)
            nse_v = np.median([nse_kge_dias(fv["y"].to_numpy()[i], pv[i])[0]
                               for i in fv.groupby("gauge_id").indices.values()])
            print(f"lgbm_{conf}_s{s}: NSE mediano de validación {nse_v:.4f} (8.4: {ref[VALIDACION_8_4[(conf, s)]]:.4f})")


def predecir_lgbm_final():
    """Variables de los días de prueba con caudal de las 183 cuencas de prueba, con las mismas funciones de 8.4 sobre la
    serie completa de cada cuenca, y predicciones de los 6 modelos y de T4 (8.3)."""
    columnas = json.loads((DIR_SELECCION / "columnas.json").read_text(encoding="utf-8"))
    base = tabla_con_era5l()
    base["gauge_id"] = base["gauge_id"].astype(str)
    base = base[base["gauge_id"].isin(se_evaluan)]
    oni, est = leer_oni(), bloques_estaticos()
    partes = []
    for g, df in base.groupby("gauge_id", sort=True):
        df = df.set_index("date").sort_index()
        m = ((df["periodo"] == "test") & df[OBJETIVO].notna()).to_numpy()
        if not m.any():
            continue
        din = bloques_dinamicos(df, oni)
        x = pd.concat([pd.DataFrame(din[b]) for b in din], axis=1)[m]
        x = x.assign(**{col: t.loc[g, col] for t in est.values() for col in t.columns})
        x.insert(0, "obs", df.loc[m, OBJETIVO].to_numpy())
        x.insert(0, "gauge_id", g)
        partes.append(x.rename_axis("date").reset_index())
    del base
    X = pd.concat(partes, ignore_index=True)
    pred = X[["gauge_id", "date", "obs"]].copy()
    for conf in LGBM_FINAL:
        M = X[columnas_lgbm_final(conf, columnas)].to_numpy("float32")
        for s in SEMILLAS_CONFIRMACION:
            modelo = lgb.Booster(model_file=str(DIR_PRUEBA_FINAL / f"lgbm_{conf}_s{s}.txt"))
            pred[f"{conf}_s{s}"] = np.clip(modelo.predict(M), 0, None)
    t4 = lgb.Booster(model_file=str(archivo_lgbm))
    pred["t4"] = np.clip(t4.predict(X[t4.feature_name()].to_numpy("float32")), 0, None)
    pred.to_parquet(DIR_PRUEBA_FINAL / "lgbm_prueba.parquet", index=False)
    print(f"Predicciones de prueba: {len(pred):,} días de {pred['gauge_id'].nunique()} cuencas")


def ensamble_prueba(conf):
    """Promedio de las simulaciones de prueba de las 3 semillas de 8.6 (época 3)."""
    series_c = [evaluar(corrida_confirmacion(conf, s), epoca=EPOCAS_CONFIRMACION) for s in SEMILLAS_CONFIRMACION]
    return {g: series_c[0][g].assign(sim=np.mean([sc[g]["sim"].to_numpy() for sc in series_c], axis=0))
            for g in series_c[0]}


if EJECUTAR_PRUEBA_FINAL:
    entrenar_lgbm_final()
    predecir_lgbm_final()

prueba_lista = (series_temporal and (DIR_PRUEBA_FINAL / "lgbm_prueba.parquet").exists() and all(
    r is not None and (EJECUTAR_PRUEBA_FINAL or (r / "test" / f"model_epoch{EPOCAS_CONFIRMACION:03d}" / "test_results.p").exists())
    for r in corridas_conf.values()))
if prueba_lista:
    lstm_prueba = {c: ensamble_prueba(c) for c in ("base", "era5l")}
    cuencas_prueba = sorted(set(series_temporal) & set(lstm_prueba["base"]) & set(lstm_prueba["era5l"]))
    mascara = {g: series_temporal[g]["sim"].notna() & lstm_prueba["base"][g]["sim"].notna()
               & lstm_prueba["era5l"][g]["sim"].notna() for g in cuencas_prueba}
    pred = pd.read_parquet(DIR_PRUEBA_FINAL / "lgbm_prueba.parquet")
    pred["gauge_id"] = pred["gauge_id"].astype(str)
    for conf in LGBM_FINAL:
        pred[f"lgbm_{conf}"] = pred[[f"{conf}_s{s}" for s in SEMILLAS_CONFIRMACION]].mean(axis=1)
    lgbm_prueba = {g: t.set_index("date") for g, t in pred.groupby("gauge_id")}

    def lgbm_en_mismos_dias(g, col):
        s = series_temporal[g][["obs"]].copy()
        s["sim"] = lgbm_prueba[g][col].reindex(s.index) if g in lgbm_prueba else np.nan
        return s.where(mascara[g], np.nan)

    def lstm_en_mismos_dias(series, g):
        return series[g].where(mascara[g], np.nan)

    def promedio_en_mismos_dias(series, col, g):
        a, b = lstm_en_mismos_dias(series, g), lgbm_en_mismos_dias(g, col)
        return a.assign(sim=(a["sim"] + b["sim"]) / 2)

    modelos_prueba = {
        "T1: LSTM, 1 semilla": {g: lstm_en_mismos_dias(series_temporal, g) for g in cuencas_prueba},
        "T4: LightGBM": {g: lgbm_en_mismos_dias(g, "t4") for g in cuencas_prueba},
        "T5: T1 + T4": {g: promedio_en_mismos_dias(series_temporal, "t4", g) for g in cuencas_prueba},
        "C1: LSTM actual, 3 semillas": {g: lstm_en_mismos_dias(lstm_prueba["base"], g) for g in cuencas_prueba},
        "LightGBM actual, 3 semillas": {g: lgbm_en_mismos_dias(g, "lgbm_base") for g in cuencas_prueba},
        "C1 + LightGBM actual": {g: promedio_en_mismos_dias(lstm_prueba["base"], "lgbm_base", g) for g in cuencas_prueba},
        "C2: LSTM con ERA5-Land, 3 semillas": {g: lstm_en_mismos_dias(lstm_prueba["era5l"], g) for g in cuencas_prueba},
        "LightGBM con ERA5-Land, 3 semillas": {g: lgbm_en_mismos_dias(g, "lgbm_era5l") for g in cuencas_prueba},
        "FINAL: C2 + LightGBM con ERA5-Land": {g: promedio_en_mismos_dias(lstm_prueba["era5l"], "lgbm_era5l", g)
                                               for g in cuencas_prueba},
        "LightGBM con ERA5-Land, todos los días con caudal": {g: t[["obs"]].assign(sim=t["lgbm_era5l"])
                                                              for g, t in lgbm_prueba.items()},
    }
    metricas_prueba = {k: metricas(v) for k, v in modelos_prueba.items()}
    filas_prueba = {}
    for k, t in metricas_prueba.items():
        fila = resumir(t)[["cuencas", "NSE", "KGE", "FHV", "FLV", "cuencas con NSE < 0"]]
        fila["NSE ≥ 0,5"], fila["NSE ≥ 0,7"] = int((t["NSE"] >= 0.5).sum()), int((t["NSE"] >= 0.7).sum())
        for r, v in t["NSE"].groupby(region_de.reindex(t.index)).median().items():
            fila[f"NSE {r}"] = round(v, 3)
        filas_prueba[k] = fila
    print(f"Días en que T1 y C2 difieren en tener simulación: "
          f"{sum(int((series_temporal[g]['sim'].notna() != lstm_prueba['era5l'][g]['sim'].notna()).sum()) for g in cuencas_prueba)}")
    display(pd.DataFrame(filas_prueba).T)
    pares = [("FINAL: C2 + LightGBM con ERA5-Land", "T5: T1 + T4"),
             ("FINAL: C2 + LightGBM con ERA5-Land", "C1 + LightGBM actual"),
             ("C2: LSTM con ERA5-Land, 3 semillas", "C1: LSTM actual, 3 semillas"),
             ("C1: LSTM actual, 3 semillas", "T1: LSTM, 1 semilla"),
             ("LightGBM con ERA5-Land, 3 semillas", "LightGBM actual, 3 semillas"),
             ("FINAL: C2 + LightGBM con ERA5-Land", "C2: LSTM con ERA5-Land, 3 semillas")]
    comparaciones_prueba = pd.DataFrame([{**diferencia_por_cuenca(f"{a} − {b}", metricas_prueba[b][m],
                                                                 metricas_prueba[a][m]), "métrica": m}
                                         for a, b in pares for m in ("NSE", "KGE")])
    print("Diferencias por cuenca (a = el segundo modelo de cada par, b = el primero):")
    display(comparaciones_prueba.set_index(["comparación", "métrica"]).round(4))
    series_final_era5l = modelos_prueba["FINAL: C2 + LightGBM con ERA5-Land"]
    if EJECUTAR_PRUEBA_FINAL:
        pd.DataFrame(filas_prueba).T.to_csv(DIR_PRUEBA_FINAL / "resumen.csv", encoding="utf-8-sig")
        pd.concat(series_final_era5l, names=["gauge_id"]).reset_index().dropna(subset=["sim"]).to_parquet(
            DIR_PRUEBA_FINAL / "series_final.parquet", index=False)
else:
    print("Evaluación final pendiente: hacen falta las corridas de 8.6 y las predicciones de LightGBM")

Días en que T1 y C2 difieren en tener simulación: 0


,cuencas,NSE,KGE,FHV,FLV,cuencas con NSE < 0,"NSE ≥ 0,5","NSE ≥ 0,7",NSE Amazon,NSE Caribbean,NSE Magdalena-Cauca,NSE Orinoco,NSE Pacific
"T1: LSTM, 1 semilla",183.0,0.394,0.513,-29.752,8.830,20.0,56.0,21.0,0.608,0.323,0.321,0.705,0.420
T4: LightGBM,183.0,0.390,0.525,-29.511,17.897,20.0,52.0,21.0,0.615,0.354,0.359,0.702,0.429
T5: T1 + T4,183.0,0.410,0.518,-29.777,24.115,20.0,59.0,21.0,0.635,0.353,0.360,0.686,0.432
"C1: LSTM actual, 3 semillas",183.0,0.389,0.511,-30.556,24.687,20.0,58.0,18.0,0.635,0.335,0.354,0.669,0.380
"LightGBM actual, 3 semillas",183.0,0.396,0.525,-29.653,21.471,20.0,54.0,21.0,0.620,0.353,0.360,0.705,0.426
C1 + LightGBM actual,183.0,0.415,0.534,-30.745,28.194,19.0,63.0,19.0,0.643,0.358,0.362,0.689,0.432
"C2: LSTM con ERA5-Land, 3 semillas",183.0,0.432,0.539,-28.478,22.994,14.0,72.0,23.0,0.704,0.371,0.384,0.716,0.502
"LightGBM con ERA5-Land, 3 semillas",183.0,0.417,0.529,-29.607,22.965,18.0,64.0,21.0,0.656,0.349,0.371,0.710,0.489
FINAL: C2 + LightGBM con ERA5-Land,183.0,0.437,0.536,-29.521,27.368,17.0,72.0,22.0,0.699,0.376,0.381,0.711,0.501
"LightGBM con ERA5-Land, todos los días con caudal",183.0,0.412,0.518,-29.299,29.457,16.0,59.0,20.0,0.656,0.285,0.375,0.703,0.474


Diferencias por cuenca (a = el segundo modelo de cada par, b = el primero):


mediana a  mediana b  dif. mediana por cuenca  cuencas que mejoran  cuencas  \
comparación                                        métrica                                                                                
FINAL: C2 + LightGBM con ERA5-Land − T5: T1 + T4   NSE         0.4099     0.4371                   0.0199                  127      183   
                                                   KGE         0.5182     0.5356                   0.0016                   95      183   
FINAL: C2 + LightGBM con ERA5-Land − C1 + Light... NSE         0.4148     0.4371                   0.0204                  133      183   
                                                   KGE         0.5341     0.5356                   0.0047                  101      183   
C2: LSTM con ERA5-Land, 3 semillas − C1: LSTM a... NSE         0.3887     0.4319                   0.0240                  120      183   
                                                   KGE         0.5110     0.5387                   0.0090                  110      183   
C1: LSTM actual, 3 semillas − T1: LSTM, 1 semilla  NSE         0.3939     0.3887                   0.0084                  110      183   
                                                   KGE         0.5130     0.5110                  -0.0006                   90      183   
LightGBM con ERA5-Land, 3 semillas − LightGBM a... NSE         0.3960     0.4174                   0.0153                  126      183   
                                                   KGE         0.5255     0.5293                  -0.0009                   90      183   
FINAL: C2 + LightGBM con ERA5-Land − C2: LSTM c... NSE         0.4319     0.4371                   0.0018                  103      183   
                                                   KGE         0.5387     0.5356                  -0.0047                   76      183   

                                                            p (Wilcoxon)  
comparación                                        métrica                
FINAL: C2 + LightGBM con ERA5-Land − T5: T1 + T4   NSE            0.0000  
                                                   KGE            0.3754  
FINAL: C2 + LightGBM con ERA5-Land − C1 + Light... NSE            0.0000  
                                                   KGE            0.0786  
C2: LSTM con ERA5-Land, 3 semillas − C1: LSTM a... NSE            0.0000  
                                                   KGE            0.0205  
C1: LSTM actual, 3 semillas − T1: LSTM, 1 semilla  NSE            0.0126  
                                                   KGE            0.9489  
LightGBM con ERA5-Land, 3 semillas − LightGBM a... NSE            0.0000  
                                                   KGE            0.8420  
FINAL: C2 + LightGBM con ERA5-Land − C2: LSTM c... NSE            0.0370  
                                                   KGE            0.0526

## 9. Etapa 3: flujo espacial (cuenca no vista)

Validación cruzada con las 5 particiones de 5.4. En la corrida *k*:
- se entrena con las 229 cuencas fuera de la partición *k* (1981–2010), y las que tienen al menos un año de datos en 2011–2015 deciden la parada temprana;
- se simulan las 19 cuencas de la partición *k* en todo su registro.

Al final, cada una de las 95 cuencas retenidas tiene una simulación hecha por un modelo que nunca vio su caudal ni el de ninguna cuenca conectada con ella. El primer año (1981) no se simula, porque el LSTM necesita los 365 días anteriores.

Para entrenar, cambia `EJECUTAR_FLUJO_ESPACIAL` a `True`. Si se corta, al volver a ejecutar la celda se saltan las particiones terminadas y se retoma la interrumpida.

In [21]:
EJECUTAR_FLUJO_ESPACIAL = False
REGISTRO_COMPLETO = (fechas("train")[0], fechas("test")[1])


def config_particion(k):
    return {**config_base, "experiment_name": f"camels_col_espacial_h{HIDDEN_SIZE}_p{k}", "hidden_size": HIDDEN_SIZE,
            "train_basin_file": f"configs/basins/particion_{k}_entrenamiento.txt",
            "validation_basin_file": f"configs/basins/particion_{k}_validacion.txt",
            "test_basin_file": f"configs/basins/particion_{k}_prueba.txt",
            "test_start_date": REGISTRO_COMPLETO[0], "test_end_date": REGISTRO_COMPLETO[1],
            "validate_n_random_basins": len(validan_por_particion[k])}


series_espacial, estado = {}, {}
for k in range(1, N_PARTICIONES + 1):
    run_dir = entrenar(config_particion(k)) if EJECUTAR_FLUJO_ESPACIAL else corrida_terminada(config_particion(k))
    estado[f"partición {k}"] = run_dir.name if run_dir is not None else "pendiente"
    if run_dir is not None:
        series_espacial.update(evaluar(run_dir))

display(pd.Series(estado, name="corrida"))
if series_espacial:
    metricas_espacial = metricas(series_espacial)
    display(resumir(metricas_espacial))

partición 1    pendiente
partición 2    pendiente
partición 3    pendiente
partición 4    pendiente
partición 5    pendiente
Name: corrida, dtype: object

## 10. Baselines

Sin un referente no se puede decir si un NSE de 0,70 es bueno. Se usan dos baselines que no requieren entrenamiento, uno por flujo:

- **Flujo temporal: climatología de la cuenca.** Para cada día del año, el caudal medio de ese día en 1981–2010, suavizado con una media móvil de 31 días. Repite el año promedio: capta la estacionalidad, pero no las diferencias entre años ni los eventos. Usa el caudal observado de la cuenca, así que solo sirve para cuencas aforadas. Se evalúa en las mismas 183 cuencas que el LSTM.
- **Flujo espacial: climatología de 5 donantes.** Para cada cuenca retenida, el promedio de las climatologías de las 5 cuencas de entrenamiento de su partición más parecidas en los 48 atributos estáticos (distancia euclidiana, con los atributos estandarizados con las cuencas de entrenamiento). Es la regionalización por similitud física, un método clásico para cuencas no aforadas, y no usa el caudal de la cuenca retenida.

En la sección 11 las métricas de cada baseline se calculan en los mismos días que las del LSTM.

In [22]:
caudal = diario.pivot(index="date", columns="gauge_id", values=OBJETIVO)
caudal.columns = caudal.columns.astype(str)
INICIO_TRAIN, FIN_TRAIN = PERIODOS["train"]
INICIO_TEST, FIN_TEST = PERIODOS["test"]
DIAS_DEL_ANIO = pd.date_range("2000-01-01", "2000-12-31").strftime("%m-%d")   # 366 días: 2000 es bisiesto
N_DONANTES = 5


def climatologia(q):
    """Caudal medio de cada día del año, suavizado con una media móvil circular de 31 días."""
    media = q.groupby(q.index.strftime("%m-%d")).mean().reindex(DIAS_DEL_ANIO)
    circular = pd.concat([media.iloc[-15:], media, media.iloc[:15]])
    return circular.rolling(31, center=True, min_periods=1).mean().iloc[15:-15]


def series_climatologia(climatologias, desde, hasta):
    """Series obs/sim en las que la simulación es la climatología de cada cuenca."""
    q = caudal.loc[desde:hasta]
    claves = q.index.strftime("%m-%d")
    return {g: pd.DataFrame({"obs": q[g].to_numpy(), "sim": climatologias[g].loc[claves].to_numpy()},
                            index=q.index.rename("date"))
            for g in climatologias.columns}


clim = pd.DataFrame({g: climatologia(caudal.loc[INICIO_TRAIN:FIN_TRAIN, g]) for g in cuencas})
base_temporal = series_climatologia(clim[se_evaluan], INICIO_TEST, FIN_TEST)   # las mismas cuencas que el LSTM (1.2)

clim_donantes = {}
for k in range(1, N_PARTICIONES + 1):
    prueba = cuenca_sistema.loc[cuenca_sistema["particion"] == k, "gauge_id"]
    entrenamiento = cuenca_sistema.loc[cuenca_sistema["particion"] != k, "gauge_id"]
    referencia = atributos_modelo.loc[entrenamiento]
    z = (atributos_modelo - referencia.mean()) / referencia.std()
    for g in prueba:
        distancia = ((z.loc[entrenamiento] - z.loc[g]) ** 2).sum(axis=1)
        clim_donantes[g] = clim[distancia.nsmallest(N_DONANTES).index].mean(axis=1)
base_espacial = series_climatologia(pd.DataFrame(clim_donantes), INICIO_TRAIN, FIN_TEST)
print(f"Climatología de {clim.shape[1]} cuencas; climatología de donantes para {len(clim_donantes)} cuencas retenidas")

Climatología de 248 cuencas; climatología de donantes para 95 cuencas retenidas


## 11. Resultados

Medianas entre cuencas por flujo, modelo y período. Las filas del LSTM aparecen cuando su corrida está terminada.

- **Retenidas** son las 95 cuencas que el flujo espacial evalúa como no aforadas.
- La fila del LSTM temporal en las retenidas compara, en las mismas cuencas y los mismos años (2016–2022), un modelo que vio su caudal de 1981 a 2010 con uno que nunca lo vio. La diferencia mide cuánto se pierde sin estación de aforo.
- El modelo final del flujo temporal, con ERA5-Land, se evalúa en 8.7 (NSE mediano de prueba 0,437). Esta tabla mantiene el LSTM de la sección 8 como referencia.
- Metas (propuesta): NSE y KGE medianos ≥ 0,70 en el flujo temporal, y NSE mediano ≥ 0,50 en el espacial.

In [23]:
cuencas_retenidas = set(cuenca_sistema.loc[cuenca_sistema["particion"] > 0, "gauge_id"])


def solo(series, conjunto):
    return {c: df for c, df in series.items() if c in conjunto}


def mismos_dias(series, referencia):
    """Recorta `series` a los días en que el LSTM (`referencia`) tiene simulación. Sin referencia, no cambia nada."""
    if not referencia:
        return series
    return {c: df[referencia[c]["sim"].reindex(df.index).notna()] for c, df in series.items() if c in referencia}


filas_resultados = {
    ("Temporal", "LSTM", "2016–2022", "todas"): (series_temporal, INICIO_TEST, FIN_TEST),
    ("Temporal", "Climatología de la cuenca", "2016–2022", "todas"):
        (mismos_dias(base_temporal, series_temporal), INICIO_TEST, FIN_TEST),
    ("Temporal", "LSTM", "2016–2022", "retenidas"): (solo(series_temporal, cuencas_retenidas), INICIO_TEST, FIN_TEST),
    ("Espacial", "LSTM", "2016–2022", "retenidas"): (series_espacial, INICIO_TEST, FIN_TEST),
    ("Espacial", "LSTM", "registro completo", "retenidas"): (series_espacial, None, None),
    ("Espacial", f"Climatología de {N_DONANTES} donantes", "registro completo", "retenidas"):
        (mismos_dias(base_espacial, series_espacial), None, None),
}
resultados = {}
for clave, (series, desde, hasta) in filas_resultados.items():
    tabla_cuencas = metricas(series, desde, hasta) if series else pd.DataFrame()
    if len(tabla_cuencas):
        resultados[clave] = resumir(tabla_cuencas)
resultados = pd.DataFrame(resultados).T.rename_axis(["flujo", "modelo", "período", "cuencas evaluadas"])
resultados["cuencas"] = resultados["cuencas"].astype(int)
resultados["cuencas con NSE < 0"] = resultados["cuencas con NSE < 0"].astype(int)
resultados

cuencas    NSE    KGE   RMSE    MAE     FHV     FLV     FMS  \
flujo    modelo                     período           cuencas evaluadas                                                                
Temporal LSTM                       2016–2022         todas                  183  0.394  0.513  1.688  1.063 -29.752   8.830 -10.311   
         Climatología de la cuenca  2016–2022         todas                  183  0.181  0.209  1.814  1.234 -57.258  75.410 -40.003   
         LSTM                       2016–2022         retenidas               62  0.428  0.554  1.765  1.132 -29.195   5.148 -15.545   
Espacial Climatología de 5 donantes registro completo retenidas               95 -0.076 -0.044  2.828  1.957 -61.271  91.265 -59.464   

                                                                         cuencas con NSE < 0  
flujo    modelo                     período           cuencas evaluadas                       
Temporal LSTM                       2016–2022         todas                               20  
         Climatología de la cuenca  2016–2022         todas                               28  
         LSTM                       2016–2022         retenidas                            6  
Espacial Climatología de 5 donantes registro completo retenidas                           58

## 12. Fase auxiliar: transfer learning con embeddings preentrenados

Si alcanza el tiempo, después de la sección 11. Un modelo preentrenado y **congelado** resume cada cuenca en un vector (embedding). Los embeddings se calculan **una sola vez** y se guardan como una tabla; lo único que se entrena es el modelo que predice el caudal a partir de ellos y de las forzantes. La pregunta es si los embeddings mejoran el flujo espacial respecto a las secciones 8 y 9.

**Diseño**

1. **Extracción, una vez:** el embedding promedio dentro del polígono de cada cuenca (los de 5.1). Con AlphaEarth son 64 números por cuenca.
2. **Reducción:** con 248 cuencas, 64 atributos más permitirían que el modelo reconozca cada cuenca en vez de aprender de sus propiedades. Se reducen a unos pocos componentes principales (PCA), ajustados solo con las cuencas de entrenamiento de cada partición.
3. **Modelo:** el mismo LSTM con su cabeza de regresión y la misma configuración, con los componentes agregados a los 48 atributos estáticos. Solo cambia la entrada, así que la diferencia en las métricas se debe a los embeddings.
4. **Evaluación:** las mismas 5 particiones y los mismos períodos. Se compara el NSE mediano de las cuencas retenidas con y sin embeddings.

**Modelos preentrenados candidatos**

| Modelo | Qué resume | Embedding | Cómo se obtiene | Costo para nosotros | Evidencia en hidrología | Ajuste a nuestros datos |
|---|---|---|---|---|---|---|
| **AlphaEarth Foundations** (Google DeepMind) | La superficie terrestre en un año, a partir de imágenes ópticas y de radar, LiDAR, simulaciones climáticas y otras fuentes | 64 dimensiones, píxeles de 10 m, un mapa por año desde 2017 | Ya calculado en Earth Engine (`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`). El promedio por polígono se calcula en el servidor | Bajo: una cuenta de Earth Engine y un script corto. No hay que descargar imágenes ni usar la GPU | Mejoró la predicción en cuencas no aforadas frente a los atributos clásicos (Qu et al., 2026). Funciona mejor combinado con ellos (Heudorfer et al., 2026) | **Alto: primera opción** |
| **TESSERA** (Universidad de Cambridge) | Series de tiempo de un año de Sentinel-1 y Sentinel-2 | 128 dimensiones, 10 m, por año (2017–2025 según su repositorio) | Ya calculado y abierto. Se descarga por teselas de 0,1° con `geotessera` | Medio-alto: las cuencas grandes cubren miles de teselas, así que habría que muestrear puntos. `geotessera` pide Python 3.12 o superior y el kernel es 3.11 | No encontrada. Sus autores reportan resultados iguales o mejores que AlphaEarth en otras tareas | Medio: alternativa abierta a AlphaEarth |
| **Copernicus-Embed** (de Copernicus-FM) | Todas las misiones Sentinel fusionadas | 768 dimensiones, celdas de 0,25° (unos 28 km), solo 2021 | Un archivo GeoTIFF global en Hugging Face | Bajo | No encontrada | Medio-bajo: una celda es más grande que muchas cuencas (la menor tiene 25 km²) |
| **Major TOM** (ESA Φ-lab) | Imágenes Sentinel-1 y Sentinel-2 de una fecha, codificadas con modelos de visión (SSL4EO, DINOv2, SigLIP) | Un vector por celda de la rejilla Major TOM | Ya calculado (Hugging Face y Copernicus Data Space) | Medio: archivos grandes y una sola fecha por celda | No encontrada | Medio-bajo |
| **SatCLIP** (Microsoft) | Una ubicación (latitud y longitud), con lo aprendido de imágenes Sentinel-2 | 256 dimensiones por punto | Modelo abierto: se evalúa en el aforo o el centroide de cada cuenca | Muy bajo | No encontrada | Bajo: equivale casi a dar la ubicación, lo que favorece memorizar regiones |
| **Presto, Galileo** (NASA Harvest) | Series de píxeles con Sentinel-1/2, clima y relieve | Un vector por píxel y período | Hay que armar las entradas en Earth Engine y ejecutar el modelo | Alto | No encontrada | Medio |
| **Modelos fundacionales de series de tiempo** (MOMENT, Chronos) aplicados a las **forzantes** | La ventana reciente de precipitación, evapotranspiración y temperaturas (nunca el caudal) | Un vector por cuenca y día | Ejecutarlos sobre unos 3,8 millones de ventanas | Alto: horas de GPU y decenas de GB | En CAMELS-US, los modelos con todas las forzantes superaron a Chronos y Sundial (He, Fox et al.) | Bajo |
| **Modelos fundacionales del tiempo atmosférico** (Aurora, Prithvi WxC) | El estado global de la atmósfera cada día | Una representación latente por día | Requieren campos globales diarios de ERA5 o MERRA-2, y tienen entre 1300 y 2300 millones de parámetros | Inviable en una GPU de 6 GB | Con Aurora congelado, la escorrentía fue la variable que peor se recuperó (Lehmann et al., 2025) | Bajo |

**Recomendación:** AlphaEarth como primera opción. Es el que mejor encaja: se agrega como atributos estáticos sin tocar las forzantes ni el preprocesamiento, cuesta poco y es el único con evidencia en cuencas no aforadas. Si mejora el flujo espacial, se puede repetir con TESSERA para comprobar el resultado con un modelo abierto. Los codificadores de series de tiempo y del tiempo atmosférico quedan fuera por costo y por evidencia.

**Limitación:** los embeddings describen la superficie desde 2017, pero el modelo simula desde 1981. En cuencas con mucha deforestación o expansión agrícola (Amazonía, Orinoquía), el embedding no representa bien las primeras décadas.

Fuentes: [AlphaEarth en Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL) · [Qu et al. 2026](https://arxiv.org/abs/2601.01558) · [Heudorfer et al. 2026](https://egusphere.copernicus.org/preprints/2026/egusphere-2026-1965/) · [TESSERA](https://arxiv.org/abs/2506.20380) · [geotessera](https://pypi.org/project/geotessera/) · [Copernicus-Embed](https://cybergis.github.io/rs-embed/models/copernicus) · [Major TOM](https://arxiv.org/abs/2412.05600) · [SatCLIP](https://climatechange.ai/papers/neurips2023/114) · [Presto](https://arxiv.org/abs/2304.14065) · [Galileo](https://proceedings.mlr.press/v267/tseng25a.html) · [MOMENT](https://arxiv.org/abs/2402.03885) · [He, Fox et al.](https://arxiv.org/abs/2410.15218) · [Aurora y variables hidrológicas](https://arxiv.org/abs/2506.19088) · [Prithvi WxC](https://arxiv.org/abs/2409.13598)

## 13. Próximos pasos

1. Etapa 2: flujo temporal con lote 256 (sección 8): hecho, NSE mediano 0,39 en prueba.
2. Flujo temporal con lote 64 (8.1): hecho, mismo NSE; el flujo espacial usa lote 256.
3. Modelo final del flujo temporal, entrenado con 1981–2015 (8.2): hecho, sin mejora significativa.
4. LightGBM y promedio con el LSTM (8.3): hecho, NSE mediano 0,410 en prueba.
5. ERA5-Land como entradas (8.4 a 8.7): hecho. El promedio LSTM + LightGBM, los dos con ERA5-Land, da NSE mediano 0,437 en prueba.
6. Etapa 3: flujo espacial (sección 9).
7. Análisis: distribución (CDF) del NSE por flujo, métricas por región y sesgos en caudales altos y bajos (FHV, FLV).
8. Si alcanza el tiempo: ajuste de `hidden_size` (sección 7) y fase auxiliar con AlphaEarth (sección 12).